In [1]:
# 06_RESULT — PART1 / CELL 01
# CROSS-PHASE INTEGRITY & RESULTS BLUEPRINT
#
# SOURCE READ-ONLY | NO TRAINING | NO OVERWRITE
# 348-ID RECONCILIATION | 17 FIGURE HASH CHECK
# IMMUTABLE RESULTS + SHA256 AUDIT
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import uuid

import numpy as np
import pandas as pd

ROOT = Path(r"C:\Penelitian Sentimen Bencana Nasional")

MOD = ROOT / "04_Modeling" / "outputs"
TMP = ROOT / "05_Temporal" / "outputs"
OUT = ROOT / "06_Result" / "outputs" / "cross_phase_integrity"

MODEL = MOD / "leakage_grouping" / "modeling_labeled_348_grouped.csv"
TEMPORAL = TMP / "timestamp_canonicalization" / "temporal_labeled_348_locked.csv"
OOF = MOD / "comparative_evaluation" / "paired_oof_comparison_348.csv"
METRICS = MOD / "comparative_evaluation" / "model_comparison_metrics.csv"
DAILY = TMP / "daily_aggregation" / "daily_sentiment_counts_20_days.csv"

MODEL_FINDINGS = MOD / "final_evaluation_report" / "final_research_findings.json"
TEMP_FINDINGS = TMP / "comparative_temporal_report" / "temporal_research_findings.json"
MODEL_AUDIT = MOD / "final_evaluation_report" / "part20_final_report_audit.json"
TEMP_AUDIT = TMP / "comparative_temporal_report" / "part5_comparative_audit.json"

MODEL_REG = MOD / "final_evaluation_report" / "final_figure_registry.csv"
TEMP_REG = TMP / "timeline_visualizations" / "temporal_figure_registry.csv"

MODEL_FIG_DIR = MOD / "lstm_visualizations"
TEMP_FIG_DIR = TMP / "timeline_visualizations"

RECON_OUT = OUT / "cross_phase_dataset_reconciliation_348.csv"
METRICS_OUT = OUT / "locked_model_metrics.csv"
FIGURES_OUT = OUT / "integrated_figure_registry_17.csv"
BLUEPRINT_OUT = OUT / "results_chapter_blueprint.md"
AUDIT_OUT = OUT / "part1_cross_phase_integrity_audit.json"

OUTPUTS = [
    RECON_OUT,
    METRICS_OUT,
    FIGURES_OUT,
    BLUEPRINT_OUT,
    AUDIT_OUT
]

EXPECTED = {
    MODEL: "cfa7df27079ef47bdb1b0cb526bc017aa1c8fe18dc4d91fef870a3441e2e9fb7",
    TEMPORAL: "4ade3c2d0832f929e0a70ec0ae921653abdcb2aecf715a76fe0c274e674b5267",
    OOF: "670a444614a124c1b80106aa9c733c0d6b1cb690fd0ab0f20c9841d431323b1a",
    METRICS: "712aa656e0e5cee36db6f11ac4c94b4e1295cb93f87faac71e57f521091e3c57",
    DAILY: "4e4e1fd9e8b4e25b1aef729905b81306dd55a0d8053a2b509cee5d8acf89df21",
}

CASE_COUNTS = {
    "lombok_2018": 101,
    "palu_2018": 105,
    "sumatra_2025": 142
}
LABEL_COUNTS = {
    "NETRAL": 225,
    "POSITIF": 108,
    "NEGATIF": 15
}

def file_sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def relative(path):
    return path.relative_to(ROOT).as_posix()

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def write_json(path, obj):
    path.write_text(
        json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False),
        encoding="utf-8"
    )

def load_csv(path):
    return pd.read_csv(
        path, encoding="utf-8-sig",
        dtype={"id": "string"}, keep_default_na=False
    )

print("=" * 75)
print("06_RESULT PART1 — CROSS-PHASE PREFLIGHT")
print("=" * 75)

# ------------------------------------------------------------
# A. VERIFY LOCKED SOURCES
# ------------------------------------------------------------

SOURCES = list(EXPECTED) + [
    MODEL_FINDINGS, TEMP_FINDINGS,
    MODEL_AUDIT, TEMP_AUDIT,
    MODEL_REG, TEMP_REG
]

for path in SOURCES:
    assert path.is_file(), f"Missing source: {path}"

for path in OUTPUTS:
    if path.exists():
        raise FileExistsError(
            f"Existing output; refusing overwrite: {path}"
        )

source_before = {
    relative(p): file_sha(p)
    for p in SOURCES
}

for path, expected_sha in EXPECTED.items():
    assert file_sha(path) == expected_sha, (
        f"Locked SHA256 mismatch: {relative(path)}"
    )

model_audit = read_json(MODEL_AUDIT)
temporal_audit = read_json(TEMP_AUDIT)
model_findings = read_json(MODEL_FINDINGS)
temporal_findings = read_json(TEMP_FINDINGS)

assert model_audit["status"] == "PART20_FINAL_REPORT_AUDIT_PASS"
assert temporal_audit["status"] == "TEMPORAL_PART5_COMPARATIVE_AUDIT_PASS"
assert model_findings["dataset_rows"] == 348
assert temporal_findings["sample_size"] == 348
assert temporal_findings["comparison_type"] == "descriptive"

print("LOCKED SOURCE HASHES AND FINAL AUDITS: PASS")

# ------------------------------------------------------------
# B. RECONCILE ALL 348 IDs, CASES, LABELS, GROUPS
# ------------------------------------------------------------

model = load_csv(MODEL)
temporal = load_csv(TEMPORAL)
oof = load_csv(OOF)

assert len(model) == len(temporal) == len(oof) == 348

for df, name in [
    (model, "model"),
    (temporal, "temporal"),
    (oof, "oof")
]:
    assert df["id"].notna().all(), f"Missing ID: {name}"
    assert df["id"].is_unique, f"Duplicate IDs: {name}"

assert model["case"].value_counts().to_dict() == CASE_COUNTS
assert temporal["case"].value_counts().to_dict() == CASE_COUNTS
assert model["sentiment_label"].value_counts().to_dict() == LABEL_COUNTS
assert temporal["sentiment_label"].value_counts().to_dict() == LABEL_COUNTS

base = model[[
    "id", "case", "sentiment_label", "leakage_group_id"
]].copy()

base = base.rename(columns={
    "case": "case_model",
    "sentiment_label": "label_model",
    "leakage_group_id": "group_model"
})

temp_sub = temporal[[
    "id", "case", "sentiment_label",
    "leakage_group_id", "date_wib"
]].rename(columns={
    "case": "case_temporal",
    "sentiment_label": "label_temporal",
    "leakage_group_id": "group_temporal"
})

oof_sub = oof[[
    "id", "case", "true_label",
    "leakage_group_id", "validation_fold",
    "predicted_label_baseline", "predicted_label_lstm"
]].rename(columns={
    "case": "case_oof",
    "true_label": "label_oof",
    "leakage_group_id": "group_oof"
})

recon = base.merge(
    temp_sub, on="id", how="outer",
    indicator="merge_model_temporal",
    validate="one_to_one"
)

recon = recon.merge(
    oof_sub, on="id", how="outer",
    indicator="merge_with_oof",
    validate="one_to_one"
)

assert len(recon) == 348
assert recon["merge_model_temporal"].eq("both").all()
assert recon["merge_with_oof"].eq("both").all()

checks = {
    "case_match": ("case_model", "case_temporal", "case_oof"),
    "label_match": ("label_model", "label_temporal", "label_oof"),
    "group_match": ("group_model", "group_temporal", "group_oof")
}

for check_name, columns in checks.items():
    recon[check_name] = (
        recon[columns[0]].astype(str).eq(
            recon[columns[1]].astype(str)
        )
        &
        recon[columns[0]].astype(str).eq(
            recon[columns[2]].astype(str)
        )
    )
    assert recon[check_name].all(), (
        f"Cross-phase mismatch: {check_name}"
    )

assert recon["date_wib"].astype(str).str.fullmatch(
    r"\d{4}-\d{2}-\d{2}"
).all()

assert recon["validation_fold"].nunique() == 3

assert recon["predicted_label_baseline"].isin(
    LABEL_COUNTS
).all()

assert recon["predicted_label_lstm"].isin(
    LABEL_COUNTS
).all()

recon = recon.sort_values(
    ["case_model", "id"],
    kind="mergesort"
).reset_index(drop=True)

print("348-ID MODELING/TEMPORAL/OOF JOIN: PASS")
print("CASE, LABEL, LEAKAGE GROUP RECONCILIATION: PASS")

# ------------------------------------------------------------
# C. VERIFY MODEL METRICS
# ------------------------------------------------------------

metrics = pd.read_csv(METRICS, encoding="utf-8-sig")

assert len(metrics) == 2
assert set(metrics["model"]) == {"baseline", "lstm"}
assert (metrics["rows"] == 348).all()

by_model = metrics.set_index("model")

for name in ["baseline", "lstm"]:
    for key in ["accuracy", "macro_f1", "macro_recall"]:
        assert np.isclose(
            float(by_model.loc[name, key]),
            float(model_findings[name][key]),
            rtol=0, atol=1e-12
        ), f"Metric inconsistency: {name}/{key}"

# Independently reconstruct accuracy from OOF predictions.
for name, pred_col in [
    ("baseline", "predicted_label_baseline"),
    ("lstm", "predicted_label_lstm")
]:
    accuracy = (
        recon[pred_col].astype(str)
        == recon["label_model"].astype(str)
    ).mean()

    assert np.isclose(
        float(accuracy),
        float(by_model.loc[name, "accuracy"]),
        rtol=0, atol=1e-12
    )

assert (
    float(by_model.loc["baseline", "macro_f1"])
    > float(by_model.loc["lstm", "macro_f1"])
)

print("BASELINE/LSTM METRICS AND OOF ACCURACY: PASS")

# ------------------------------------------------------------
# D. RECONCILE TEMPORAL AGGREGATION
# ------------------------------------------------------------

daily = pd.read_csv(DAILY, encoding="utf-8-sig")

assert len(daily) == 20
assert int(daily["total_tweets"].sum()) == 348

for case, expected in CASE_COUNTS.items():
    part = daily.loc[daily["case"] == case]
    assert int(part["total_tweets"].sum()) == expected

    matching = recon.loc[recon["case_model"] == case]
    assert len(matching) == expected

    for label in LABEL_COUNTS:
        actual = int(part[f"count_{label}"].sum())
        expected_count = int(
            (matching["label_model"] == label).sum()
        )
        assert actual == expected_count, (
            f"Temporal mismatch: {case}/{label}"
        )

# Exact day/label match, not just total counts.
daily_from_tweets = (
    recon.groupby(
        ["case_model", "date_wib", "label_model"]
    ).size()
)

for row in daily.itertuples(index=False):
    for label in LABEL_COUNTS:
        expected_count = int(daily_from_tweets.get(
            (row.case, row.date_wib, label), 0
        ))
        assert int(getattr(row, f"count_{label}")) == expected_count

print("DAILY SENTIMENT RECONCILIATION: PASS")

# ------------------------------------------------------------
# E. VERIFY AND INTEGRATE 17 EXISTING FIGURES
# ------------------------------------------------------------

model_registry = pd.read_csv(
    MODEL_REG, encoding="utf-8-sig"
)
temp_registry = pd.read_csv(
    TEMP_REG, encoding="utf-8-sig"
)

assert len(model_registry) == 9
assert len(temp_registry) == 8

figure_rows = []
figure_sources = []

for row in model_registry.itertuples(index=False):
    path = MODEL_FIG_DIR / row.filename

    assert path.is_file(), f"Missing figure: {path}"
    assert file_sha(path) == row.sha256
    assert path.open("rb").read(8) == bytes.fromhex(
        "89504e470d0a1a0a"
    )

    figure_sources.append(path)
    figure_rows.append({
        "integrated_figure_id": f"MODEL_{row.figure_id}",
        "stage": "04_Modeling",
        "filename": row.filename,
        "relative_repository_path": relative(path),
        "sha256": file_sha(path),
        "usage": "modeling_results"
    })

for row in temp_registry.itertuples(index=False):
    path = TEMP_FIG_DIR / row.filename

    assert path.is_file(), f"Missing figure: {path}"
    assert file_sha(path) == row.figure_sha256
    assert path.open("rb").read(8) == bytes.fromhex(
        "89504e470d0a1a0a"
    )

    figure_sources.append(path)
    figure_rows.append({
        "integrated_figure_id": f"TEMP_{row.figure_id}",
        "stage": "05_Temporal",
        "filename": row.filename,
        "relative_repository_path": relative(path),
        "sha256": file_sha(path),
        "usage": "temporal_results"
    })

figure_registry = pd.DataFrame(figure_rows)

assert len(figure_registry) == 17
assert figure_registry["integrated_figure_id"].is_unique

figure_before = {
    relative(p): file_sha(p) for p in figure_sources
}

print("9 MODELING + 8 TEMPORAL FIGURE HASHES: PASS")

# ------------------------------------------------------------
# F. RESULTS CHAPTER BLUEPRINT
# ------------------------------------------------------------

blueprint = """# BLUEPRINT BAB IV — HASIL DAN PEMBAHASAN

**Penelitian:** Sentimen Penetapan Bencana Nasional

**Status:** Blueprint awal berdasarkan hasil audit lintas fase.

> Struktur ini dapat disesuaikan dengan proposal dan
> pedoman penulisan kampus. Belum merupakan naskah
> BAB IV final.

## 4.1 Gambaran Umum Dataset

- Jelaskan sumber dan proses seleksi data.
- Sajikan 348 tweet berlabel manusia.
- Distribusi kasus: Lombok 101, Palu 105, Sumatera 142.
- Distribusi sentimen: NETRAL 225, POSITIF 108, NEGATIF 15.
- Jelaskan 337 leakage groups dan pengendalian leakage.
- Jelaskan keterbatasan cakupan sampel.

## 4.2 Metode dan Pengaturan Evaluasi

- Stratified Group K-Fold tiga fold, seed 123.
- Baseline TF-IDF + Logistic Regression.
- Model LSTM_SMALL_V1.
- Metrik utama Out-of-Fold Macro-F1.
- Bedakan training, validation, dan OOF.
- Tidak ada independent holdout test set.

## 4.3 Hasil Pemodelan Sentimen

- Sajikan metrik kedua model.
- Baseline: accuracy 0.7040, Macro-F1 0.5236.
- LSTM: accuracy 0.5862, Macro-F1 0.4751.
- Bahas grafik hasil modeling dan confusion matrix.
- Jelaskan indikasi overfitting LSTM.
- Bahas kelemahan kedua model pada kelas NEGATIF.

## 4.4 Analisis Kesalahan Klasifikasi

- Kedua model benar: 165 tweet.
- Hanya baseline benar: 80 tweet.
- Hanya LSTM benar: 39 tweet.
- Kedua model salah: 64 tweet.
- Jelaskan perbedaan precision dan recall NEGATIF.
- Jangan menyatakan signifikansi statistik tanpa pengujian.

## 4.5 Distribusi Sentimen dan Analisis Temporal

- Gunakan label manusia, bukan prediksi model.
- Sajikan jumlah dan persentase harian per kasus.
- Tampilkan 20 hari kalender kasus.
- Sebanyak 15 hari memiliki observasi dan 5 hari kosong.
- Gunakan grafik temporal asli Part4.
- Hari relatif 0 berarti awal observasi, bukan awal bencana.

## 4.6 Pembahasan Perbandingan Kasus

- Bandingkan Lombok, Palu, dan Sumatera secara deskriptif.
- Bedakan jumlah tweet dan proporsi sentimen.
- Jelaskan variasi cakupan waktu dan distribusi sampel.
- Hindari generalisasi sebagai opini seluruh masyarakat.
- Hindari klaim hubungan sebab-akibat kebijakan.

## 4.7 Keterbatasan Penelitian

- Hanya 348 tweet berlabel.
- Kelas NEGATIF sangat kecil (15 tweet).
- Perbedaan jendela pengamatan antarkasus.
- Pengaruh strategi scraping dan seleksi sampel.
- Tidak tersedia independent holdout.
- Potensi overfitting pada LSTM.
- Hasil temporal bersifat deskriptif.

## 4.8 Ringkasan Temuan

- Baseline lebih tinggi menurut OOF Macro-F1.
- LSTM memiliki Macro-Recall lebih tinggi.
- NETRAL mendominasi sampel ketiga kasus.
- Pola temporal berbeda menurut kasus.
- Simpulan terbatas pada sampel dan metode penelitian.

## Catatan Integrasi

Modeling menjelaskan performa prediksi.
Temporal menjelaskan distribusi label manusia.
Keduanya menggunakan 348 tweet yang sama,
tetapi menjawab pertanyaan analitis berbeda.

Dokumen ini merupakan blueprint.
Finalisasi BAB IV harus disesuaikan dengan
rumusan masalah, tujuan, serta pedoman akademik.
"""

# ------------------------------------------------------------
# G. AUDIT MANIFEST
# ------------------------------------------------------------

now = datetime.now(timezone.utc).isoformat()

audit = {
    "stage": "06_Result",
    "part": "1 Cross-Phase Integrity & Blueprint",
    "generated_at_utc": now,
    "modeling_source_commit":
        "5cf80cc802f99c25fc4b714a4036beea3ae930e4",
    "temporal_source_commit":
        "7571fc02d68cdb2f81b404554df4304940e2bb07",
    "dataset_rows": 348,
    "unique_ids": 348,
    "case_match_count": int(recon["case_match"].sum()),
    "label_match_count": int(recon["label_match"].sum()),
    "leakage_group_match_count":
        int(recon["group_match"].sum()),
    "oof_rows": 348,
    "daily_rows": 20,
    "daily_tweets": 348,
    "model_figures_verified": 9,
    "temporal_figures_verified": 8,
    "total_figures_verified": 17,
    "source_sha256": source_before,
    "figure_sha256": figure_before,
    "model_training_performed": False,
    "source_modified": False,
    "results_blueprint_prepared": True,
    "output_sha256": {},
    "status": "RESULT_PART1_CROSS_PHASE_INTEGRITY_PASS"
}

# ------------------------------------------------------------
# H. IMMUTABLE EXPORT
# ------------------------------------------------------------

OUT.mkdir(parents=True, exist_ok=True)

for path in OUTPUTS:
    if path.exists():
        raise FileExistsError(
            f"Refusing to overwrite existing output: {path}"
        )

token = uuid.uuid4().hex[:10]
temps = {
    p: OUT / f".result_p1_{token}_{i}.tmp"
    for i, p in enumerate(OUTPUTS)
}
created = []

try:
    recon.to_csv(
        temps[RECON_OUT],
        index=False, encoding="utf-8-sig"
    )

    metrics.to_csv(
        temps[METRICS_OUT],
        index=False, encoding="utf-8-sig"
    )

    figure_registry.to_csv(
        temps[FIGURES_OUT],
        index=False, encoding="utf-8-sig"
    )

    temps[BLUEPRINT_OUT].write_text(
        blueprint, encoding="utf-8"
    )

    audit["output_sha256"] = {
        p.name: file_sha(temps[p])
        for p in OUTPUTS[:-1]
    }

    write_json(temps[AUDIT_OUT], audit)

    # Check no source changed before publication.
    assert {
        relative(p): file_sha(p)
        for p in SOURCES
    } == source_before

    assert {
        relative(p): file_sha(p)
        for p in figure_sources
    } == figure_before

    # Re-open saved outputs and verify row counts.
    assert len(pd.read_csv(temps[RECON_OUT])) == 348
    assert len(pd.read_csv(temps[FIGURES_OUT])) == 17

    for destination in OUTPUTS:
        with destination.open("xb") as f:
            f.write(temps[destination].read_bytes())
        created.append(destination)

except Exception:
    # Remove only outputs created by this run.
    for path in created:
        if path.exists():
            path.unlink()
    raise

finally:
    for path in temps.values():
        if path.exists():
            path.unlink()

# ------------------------------------------------------------
# I. FINAL VERIFICATION
# ------------------------------------------------------------

for name, expected_sha in audit["output_sha256"].items():
    assert file_sha(OUT / name) == expected_sha

assert {
    relative(p): file_sha(p)
    for p in SOURCES
} == source_before

assert {
    relative(p): file_sha(p)
    for p in figure_sources
} == figure_before

assert read_json(AUDIT_OUT)["status"] == (
    "RESULT_PART1_CROSS_PHASE_INTEGRITY_PASS"
)

print("\n" + "=" * 75)
print("06_RESULT PART1 — FINAL AUDIT")
print("=" * 75)

print("Cross-phase rows     :", len(recon))
print("Matching IDs         :", recon["id"].nunique())
print("Matching cases       :", int(recon["case_match"].sum()))
print("Matching labels      :", int(recon["label_match"].sum()))
print("Matching groups      :", int(recon["group_match"].sum()))
print("Modeling figures     :", 9)
print("Temporal figures     :", 8)
print("Total figures        :", len(figure_registry))
print("Source modified      : NO")
print("Training performed   : NO")

print("\nLOCKED MODEL METRICS")
display(metrics)

print("\nINTEGRITY CHECK COUNTS")
display(
    recon[[
        "case_match", "label_match", "group_match"
    ]].sum().to_frame("matching_rows")
)

print("\nARTIFACTS:")
for p in OUTPUTS:
    print("-", p)

print("\nFINAL STATUS:", audit["status"])
print("=" * 75)


06_RESULT PART1 — CROSS-PHASE PREFLIGHT
LOCKED SOURCE HASHES AND FINAL AUDITS: PASS
348-ID MODELING/TEMPORAL/OOF JOIN: PASS
CASE, LABEL, LEAKAGE GROUP RECONCILIATION: PASS
BASELINE/LSTM METRICS AND OOF ACCURACY: PASS
DAILY SENTIMENT RECONCILIATION: PASS
9 MODELING + 8 TEMPORAL FIGURE HASHES: PASS

06_RESULT PART1 — FINAL AUDIT
Cross-phase rows     : 348
Matching IDs         : 348
Matching cases       : 348
Matching labels      : 348
Matching groups      : 348
Modeling figures     : 9
Temporal figures     : 8
Total figures        : 17
Source modified      : NO
Training performed   : NO

LOCKED MODEL METRICS


,model,rows,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1
0,baseline,348,0.704023,0.776108,0.494321,0.523555,0.689845
1,lstm,348,0.586207,0.475956,0.559630,0.475130,0.617452



INTEGRITY CHECK COUNTS


,matching_rows
case_match,348
label_match,348
group_match,348



ARTIFACTS:
- C:\Penelitian Sentimen Bencana Nasional\06_Result\outputs\cross_phase_integrity\cross_phase_dataset_reconciliation_348.csv
- C:\Penelitian Sentimen Bencana Nasional\06_Result\outputs\cross_phase_integrity\locked_model_metrics.csv
- C:\Penelitian Sentimen Bencana Nasional\06_Result\outputs\cross_phase_integrity\integrated_figure_registry_17.csv
- C:\Penelitian Sentimen Bencana Nasional\06_Result\outputs\cross_phase_integrity\results_chapter_blueprint.md
- C:\Penelitian Sentimen Bencana Nasional\06_Result\outputs\cross_phase_integrity\part1_cross_phase_integrity_audit.json

FINAL STATUS: RESULT_PART1_CROSS_PHASE_INTEGRITY_PASS


In [3]:
# 06_RESULT — PART2 / CELL 02
# CONSOLIDATED RESEARCH RESULTS
#
# LOCKED SOURCES | OOF METRIC RECONCILIATION
# DATASET + MODEL + ERROR + TEMPORAL TABLES
# NO TRAINING | NO OVERWRITE | SHA256 AUDIT
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

ROOT = Path(r"C:\Penelitian Sentimen Bencana Nasional")
MOD = ROOT / "04_Modeling" / "outputs"
TEMP = ROOT / "05_Temporal" / "outputs"
P1 = ROOT / "06_Result" / "outputs" / "cross_phase_integrity"
OUT = ROOT / "06_Result" / "outputs" / "consolidated_results"

RECON = P1 / "cross_phase_dataset_reconciliation_348.csv"
METRICS = P1 / "locked_model_metrics.csv"
P1_AUDIT = P1 / "part1_cross_phase_integrity_audit.json"

OOF = MOD / "comparative_evaluation" / "paired_oof_comparison_348.csv"
MODEL_FINDINGS = MOD / "final_evaluation_report" / "final_research_findings.json"
TEMP_FINDINGS = TEMP / "comparative_temporal_report" / "temporal_research_findings.json"
DAILY = TEMP / "daily_aggregation" / "daily_sentiment_counts_20_days.csv"

DATA_OUT = OUT / "dataset_characteristics.csv"
MODEL_OUT = OUT / "consolidated_model_evaluation.csv"
ERROR_OUT = OUT / "model_error_analysis_by_class.csv"
TEMP_OUT = OUT / "consolidated_temporal_findings.csv"
REPORT_OUT = OUT / "consolidated_research_results.md"
AUDIT_OUT = OUT / "part2_consolidated_results_audit.json"

OUTPUTS = [
    DATA_OUT, MODEL_OUT, ERROR_OUT,
    TEMP_OUT, REPORT_OUT, AUDIT_OUT
]

SOURCES = [
    RECON, METRICS, P1_AUDIT,
    OOF, MODEL_FINDINGS, TEMP_FINDINGS, DAILY
]

CASES = ["lombok_2018", "palu_2018", "sumatra_2025"]
LABELS = ["NETRAL", "POSITIF", "NEGATIF"]

CASE_NAMES = {
    "lombok_2018": "Lombok 2018",
    "palu_2018": "Palu 2018",
    "sumatra_2025": "Sumatera 2025"
}

EXPECTED_CASE = {
    "lombok_2018": 101,
    "palu_2018": 105,
    "sumatra_2025": 142
}
EXPECTED_LABEL = {
    "NETRAL": 225,
    "POSITIF": 108,
    "NEGATIF": 15
}

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def write_json(path, value):
    path.write_text(
        json.dumps(
            value, indent=2,
            ensure_ascii=False, allow_nan=False
        ),
        encoding="utf-8"
    )

def read_csv(path, has_id=False):
    kwargs = {"encoding": "utf-8-sig"}
    if has_id:
        kwargs["dtype"] = {"id": "string"}
    return pd.read_csv(path, **kwargs)

def check_equal(a, b, name, atol=1e-12):
    assert np.isclose(
        float(a), float(b),
        atol=atol, rtol=0
    ), f"{name}: {a} != {b}"

def markdown_table(df, digits=4):
    """Convert DataFrame to Markdown without tabulate."""
    def fmt(value):
        if pd.isna(value):
            return ""
        if isinstance(value, (float, np.floating)):
            return f"{value:.{digits}f}"
        return str(value).replace("|", "\\|").replace("\n", " ")

    headers = [str(col) for col in df.columns]
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |"
    ]

    for row in df.itertuples(index=False, name=None):
        lines.append("| " + " | ".join(fmt(v) for v in row) + " |")

    return "\n".join(lines)

print("=" * 75)
print("RESULT PART2 — CONSOLIDATION PREFLIGHT")
print("=" * 75)

# ------------------------------------------------------------
# A. PREFLIGHT & SOURCE LOCK
# ------------------------------------------------------------

for path in SOURCES:
    assert path.is_file(), f"Missing source: {path}"

for path in OUTPUTS:
    if path.exists():
        raise FileExistsError(
            f"Existing Part2 output; refusing overwrite: {path}"
        )

source_hashes = {
    p.relative_to(ROOT).as_posix(): sha256(p)
    for p in SOURCES
}

p1_audit = read_json(P1_AUDIT)
model_findings = read_json(MODEL_FINDINGS)
temp_findings = read_json(TEMP_FINDINGS)

assert p1_audit["status"] == (
    "RESULT_PART1_CROSS_PHASE_INTEGRITY_PASS"
)

for path in [RECON, METRICS]:
    expected = p1_audit["output_sha256"][path.name]
    assert sha256(path) == expected

assert sha256(OOF) == p1_audit["source_sha256"][
    "04_Modeling/outputs/comparative_evaluation/"
    "paired_oof_comparison_348.csv"
]

assert sha256(DAILY) == p1_audit["source_sha256"][
    "05_Temporal/outputs/daily_aggregation/"
    "daily_sentiment_counts_20_days.csv"
]

assert model_findings["status"] == (
    "PART20_FINAL_FINDINGS_PREPARED"
)
assert temp_findings["status"] == (
    "TEMPORAL_PART5_FINDINGS_PREPARED"
)

print("PART1 AUDIT & SOURCE SHA256: PASS")

# ------------------------------------------------------------
# B. LOAD & CHECK INTEGRATED TWEET DATA
# ------------------------------------------------------------

recon = read_csv(RECON, has_id=True)
oof = read_csv(OOF, has_id=True)
locked = read_csv(METRICS)
daily = read_csv(DAILY)

assert len(recon) == len(oof) == 348
assert recon["id"].is_unique
assert oof["id"].is_unique
assert len(daily) == 20

assert recon["case_match"].all()
assert recon["label_match"].all()
assert recon["group_match"].all()

assert set(recon["id"]) == set(oof["id"])

check = recon[[
    "id", "label_model",
    "predicted_label_baseline",
    "predicted_label_lstm"
]].merge(
    oof[[
        "id", "true_label",
        "predicted_label_baseline",
        "predicted_label_lstm"
    ]],
    on="id", validate="one_to_one",
    suffixes=("_recon", "_oof")
)

assert (check["label_model"] == check["true_label"]).all()

for model in ["baseline", "lstm"]:
    assert (
        check[f"predicted_label_{model}_recon"]
        == check[f"predicted_label_{model}_oof"]
    ).all()

assert recon["case_model"].value_counts().to_dict() == (
    EXPECTED_CASE
)
assert recon["label_model"].value_counts().to_dict() == (
    EXPECTED_LABEL
)

print("348 TWEET-LEVEL PREDICTIONS: PASS")

# ------------------------------------------------------------
# C. DATASET CHARACTERISTICS TABLE
# ------------------------------------------------------------

dataset_rows = []

for case in CASES:
    subset = recon[recon["case_model"] == case]
    n = len(subset)

    row = {
        "case": case,
        "case_name": CASE_NAMES[case],
        "total_tweets": n,
        "sample_pct": n / len(recon) * 100,
        "unique_leakage_groups":
            subset["group_model"].nunique()
    }

    for label in LABELS:
        count = int((subset["label_model"] == label).sum())
        row[f"count_{label}"] = count
        row[f"pct_{label}"] = count / n * 100

    dataset_rows.append(row)

dataset_table = pd.DataFrame(dataset_rows)

assert dataset_table["total_tweets"].sum() == 348

for label in LABELS:
    assert int(dataset_table[f"count_{label}"].sum()) == (
        EXPECTED_LABEL[label]
    )

unique_groups = recon["group_model"].nunique()
assert unique_groups == model_findings["unique_leakage_groups"]
assert unique_groups == 337

print("DATASET CHARACTERISTICS: PASS")

# ------------------------------------------------------------
# D. RECOMPUTE MODEL PERFORMANCE FROM OOF
# ------------------------------------------------------------

locked = locked.set_index("model")
assert set(locked.index) == {"baseline", "lstm"}

model_rows = []
class_rows = []

y_true = oof["true_label"].astype(str).to_numpy()

for model in ["baseline", "lstm"]:

    y_pred = (
        oof[f"predicted_label_{model}"]
        .astype(str).to_numpy()
    )

    accuracy = accuracy_score(y_true, y_pred)

    macro_p, macro_r, macro_f1, _ = (
        precision_recall_fscore_support(
            y_true, y_pred,
            labels=LABELS,
            average="macro",
            zero_division=0
        )
    )

    weighted_p, weighted_r, weighted_f1, _ = (
        precision_recall_fscore_support(
            y_true, y_pred,
            labels=LABELS,
            average="weighted",
            zero_division=0
        )
    )

    results = {
        "model": model,
        "n_oof": len(oof),
        "accuracy": float(accuracy),
        "macro_precision": float(macro_p),
        "macro_recall": float(macro_r),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
        "evaluation": "3-fold SGKF OOF",
        "independent_holdout": False
    }

    for metric in [
        "accuracy", "macro_precision",
        "macro_recall", "macro_f1", "weighted_f1"
    ]:
        check_equal(
            results[metric],
            locked.loc[model, metric],
            f"{model}/{metric}"
        )

    model_rows.append(results)

    class_p, class_r, class_f, support = (
        precision_recall_fscore_support(
            y_true, y_pred,
            labels=LABELS,
            average=None,
            zero_division=0
        )
    )

    for i, label in enumerate(LABELS):
        tp = int(((y_true == label) & (y_pred == label)).sum())
        fp = int(((y_true != label) & (y_pred == label)).sum())
        fn = int(((y_true == label) & (y_pred != label)).sum())
        tn = int(((y_true != label) & (y_pred != label)).sum())

        assert tp + fn == int(support[i])
        assert tp + fp + fn + tn == 348

        class_rows.append({
            "model": model,
            "sentiment_label": label,
            "support": int(support[i]),
            "true_positive": tp,
            "false_positive": fp,
            "false_negative": fn,
            "true_negative": tn,
            "precision": float(class_p[i]),
            "recall": float(class_r[i]),
            "f1": float(class_f[i])
        })

model_table = pd.DataFrame(model_rows)
class_table = pd.DataFrame(class_rows)

assert len(model_table) == 2
assert len(class_table) == 6

assert (
    model_table.set_index("model").loc["baseline", "macro_f1"]
    >
    model_table.set_index("model").loc["lstm", "macro_f1"]
)

# Compare negative-class findings to Part20.
for model in ["baseline", "lstm"]:
    neg = class_table[
        (class_table["model"] == model)
        & (class_table["sentiment_label"] == "NEGATIF")
    ].iloc[0]

    reference = model_findings["negative_class"][model]

    assert int(neg["true_positive"]) == reference["true_positive"]
    assert int(neg["false_positive"]) == reference["false_positive"]
    assert int(neg["false_negative"]) == reference["false_negative"]

    for metric in ["precision", "recall", "f1"]:
        check_equal(
            neg[metric],
            reference[metric],
            f"NEGATIF/{model}/{metric}"
        )

print("OOF METRICS & NEGATIVE CLASS RECOMPUTATION: PASS")

# ------------------------------------------------------------
# E. PAIRED ERROR ANALYSIS
# ------------------------------------------------------------

baseline_correct = (
    oof["true_label"] == oof["predicted_label_baseline"]
)
lstm_correct = (
    oof["true_label"] == oof["predicted_label_lstm"]
)

paired = {
    "both_correct":
        int((baseline_correct & lstm_correct).sum()),
    "baseline_only_correct":
        int((baseline_correct & ~lstm_correct).sum()),
    "lstm_only_correct":
        int((~baseline_correct & lstm_correct).sum()),
    "both_wrong":
        int((~baseline_correct & ~lstm_correct).sum())
}

assert sum(paired.values()) == 348
assert paired == model_findings["paired_correctness"]

print("PAIRED OOF ERROR CATEGORY COUNTS: PASS")

# ------------------------------------------------------------
# F. CONSOLIDATE TEMPORAL FINDINGS
# ------------------------------------------------------------

temp_rows = []

for finding in temp_findings["case_findings"]:
    case = finding["case"]
    subset = daily[daily["case"] == case]

    count = int(subset["total_tweets"].sum())
    observed_days = int((subset["total_tweets"] > 0).sum())
    zero_days = int((subset["total_tweets"] == 0).sum())

    assert count == finding["total_tweets"]
    assert observed_days == finding["observed_days"]
    assert zero_days == finding["zero_tweet_days"]
    assert len(subset) == finding["calendar_days"]

    peak = int(subset["total_tweets"].max())
    peak_dates = sorted(
        subset.loc[
            subset["total_tweets"] == peak, "date_wib"
        ].astype(str).tolist()
    )

    assert peak == finding["peak_daily_tweet_count"]
    assert peak_dates == sorted(finding["peak_dates_wib"])

    record = {
        "case": case,
        "case_name": CASE_NAMES[case],
        "total_tweets": count,
        "first_date_wib": subset["date_wib"].min(),
        "last_date_wib": subset["date_wib"].max(),
        "calendar_days": len(subset),
        "observed_days": observed_days,
        "zero_tweet_days": zero_days,
        "peak_date_wib": "; ".join(peak_dates),
        "peak_daily_tweets": peak,
        "peak_share_pct": peak / count * 100
    }

    for label in LABELS:
        c = int(subset[f"count_{label}"].sum())

        assert c == finding["sentiment_counts"][label]

        record[f"count_{label}"] = c
        record[f"pct_{label}"] = c / count * 100

    temp_rows.append(record)

temp_table = pd.DataFrame(temp_rows)

assert len(temp_table) == 3
assert temp_table["total_tweets"].sum() == 348
assert temp_table["calendar_days"].sum() == 20
assert temp_table["observed_days"].sum() == 15
assert temp_table["zero_tweet_days"].sum() == 5

print("TEMPORAL FINDINGS RECONCILIATION: PASS")

# ------------------------------------------------------------
# G. PREPARE INTEGRATED RESEARCH REPORT
# ------------------------------------------------------------

model_view = model_table[[
    "model", "n_oof", "accuracy",
    "macro_precision", "macro_recall",
    "macro_f1", "weighted_f1"
]].copy()

data_view = dataset_table[[
    "case_name", "total_tweets",
    "count_NETRAL", "count_POSITIF", "count_NEGATIF"
]].copy()

error_view = class_table[[
    "model", "sentiment_label", "support",
    "true_positive", "false_positive",
    "false_negative", "precision", "recall", "f1"
]].copy()

temp_view = temp_table[[
    "case_name", "calendar_days",
    "observed_days", "zero_tweet_days",
    "peak_date_wib", "peak_daily_tweets"
]].copy()

b = model_table.set_index("model").loc["baseline"]
l = model_table.set_index("model").loc["lstm"]

report = f"""# HASIL PENELITIAN TERKONSOLIDASI

**Penelitian:** Sentimen Penetapan Bencana Nasional

**Tahap:** 06_Result Part2 — Consolidated Research Results

**Tanggal pembuatan UTC:** {datetime.now(timezone.utc).isoformat()}

## 1. Ruang Lingkup

Laporan ini mengintegrasikan hasil fase Modeling dan
Temporal menggunakan 348 tweet yang telah direkonsiliasi
pada Result Part1.

Perhitungan model bersumber dari prediksi out-of-fold (OOF).
Analisis temporal menggunakan label manusia yang dikunci.

Tidak dilakukan pelatihan model atau penambahan data baru.

## 2. Karakteristik Dataset

{markdown_table(data_view)}

Dataset berisi **348 tweet**, dengan distribusi label:
NETRAL **225**, POSITIF **108**, dan NEGATIF **15**.

Jumlah leakage group unik: **{unique_groups}**.
Distribusi kelas tidak seimbang; karena itu interpretasi
metrik tidak cukup hanya menggunakan accuracy.

## 3. Evaluasi Model

{markdown_table(model_view)}

Protokol evaluasi: **{model_findings["cv_configuration"]}**.
Metrik utama: **{model_findings["primary_metric"]}**.

Baseline TF-IDF + Logistic Regression memiliki
OOF Macro-F1 **{b["macro_f1"]:.4f}**, sedangkan
LSTM memiliki **{l["macro_f1"]:.4f}**.

Perbedaan Macro-F1 baseline dikurangi LSTM adalah
**{b["macro_f1"] - l["macro_f1"]:.4f}** poin.

Baseline juga memiliki accuracy yang lebih tinggi.
Sebaliknya, Macro-Recall LSTM lebih tinggi.

Peringkat tersebut bersifat deskriptif untuk hasil OOF.
Tidak ada independent holdout test set.

## 4. Analisis Kesalahan per Kelas

{markdown_table(error_view)}

Kelas NEGATIF memiliki 15 contoh aktual.
Baseline menghasilkan recall NEGATIF lebih rendah,
sementara LSTM memiliki lebih banyak false positive
NEGATIF. Perbedaan ini harus dibahas sebagai
trade-off precision dan recall, bukan hanya jumlah
prediksi benar.

## 5. Perbandingan Kesalahan Berpasangan

| Kategori | Jumlah tweet |
| --- | ---: |
| Kedua model benar | {paired["both_correct"]} |
| Hanya baseline benar | {paired["baseline_only_correct"]} |
| Hanya LSTM benar | {paired["lstm_only_correct"]} |
| Kedua model salah | {paired["both_wrong"]} |
| **Total** | **348** |

Perbandingan ini menunjukkan tumpang tindih dan
perbedaan pola kesalahan kedua model pada tweet
yang sama. Belum dilakukan uji signifikansi statistik.

## 6. Ringkasan Hasil Temporal

{markdown_table(temp_view)}

Jumlah kalender kasus: **20 hari**.
Hari dengan observasi: **15 hari**.
Hari tanpa observasi: **5 hari**.

Komposisi sentimen NETRAL merupakan yang terbesar
pada ketiga kasus. Hari tanpa tweet ditandai secara
eksplisit dan tidak dihitung sebagai persentase nol.

Perbandingan volume dan proporsi bersifat deskriptif
atas sampel yang terkumpul, bukan estimasi aktivitas
atau opini seluruh pengguna X.

## 7. Sintesis Hasil

1. Baseline mengungguli LSTM pada metrik utama
   OOF Macro-F1, tetapi LSTM memiliki Macro-Recall
   lebih tinggi.
2. Kelas NEGATIF menjadi tantangan evaluasi
   karena support sangat kecil.
3. Analisis Temporal menggambarkan dominasi NETRAL
   pada ketiga kasus dan cakupan observasi yang
   tidak seragam.
4. Hasil Modeling mengevaluasi kemampuan prediksi,
   sementara hasil Temporal menggambarkan label
   manusia menurut waktu.
5. Tidak ada bukti dalam analisis ini yang cukup
   untuk menyimpulkan hubungan kausal antara
   penetapan status bencana dan sentimen masyarakat.

## 8. Batasan Interpretasi

- Sampel penelitian terdiri atas 348 tweet.
- Label NEGATIF hanya mencakup 15 tweet.
- Evaluasi model menggunakan OOF tiga fold,
  tanpa test set independen.
- Periode pengamatan antarkasus berbeda.
- Hari tanpa tweet tidak berarti aktivitas publik nol.
- Tidak dilakukan pengujian signifikansi atas
  perbedaan model dalam tahap ini.
- Tidak dilakukan inferensi populasi atau kausal.

## 9. Kesiapan Part3

Empat tabel konsolidasi dalam laporan ini
dapat digunakan untuk menyeleksi figur yang
relevan bagi BAB IV.

**Tahap berikutnya: 06_Result Part3 —
Results Visualization & Figure Selection.**
"""

# ------------------------------------------------------------
# H. CREATE OUTPUTS WITHOUT OVERWRITING
# ------------------------------------------------------------

OUT.mkdir(parents=True, exist_ok=True)

for path in OUTPUTS:
    if path.exists():
        raise FileExistsError(
            f"Refusing to overwrite: {path}"
        )

output_bytes = {}

csv_tables = {
    DATA_OUT: dataset_table,
    MODEL_OUT: model_table,
    ERROR_OUT: class_table,
    TEMP_OUT: temp_table
}

for path, table in csv_tables.items():
    output_bytes[path] = table.to_csv(
        index=False
    ).encode("utf-8-sig")

output_bytes[REPORT_OUT] = report.encode("utf-8")

audit = {
    "stage": "06_Result",
    "part": "2 Consolidated Research Results",
    "generated_at_utc":
        datetime.now(timezone.utc).isoformat(),
    "source_commit":
        "5f591e0456d380758ef4ac3f06253ca2eabe4be3",
    "source_sha256": source_hashes,
    "dataset_rows": 348,
    "unique_leakage_groups": unique_groups,
    "model_count": 2,
    "per_class_rows": 6,
    "case_count": 3,
    "daily_calendar_rows": 20,
    "paired_correctness": paired,
    "oof_metrics_recomputed": True,
    "oof_metrics_match_locked": True,
    "temporal_findings_recomputed": True,
    "source_modified": False,
    "model_training_performed": False,
    "output_sha256": {
        p.name: hashlib.sha256(data).hexdigest()
        for p, data in output_bytes.items()
    },
    "status": "RESULT_PART2_CONSOLIDATED_RESULTS_PASS"
}

output_bytes[AUDIT_OUT] = json.dumps(
    audit,
    ensure_ascii=False, indent=2,
    allow_nan=False
).encode("utf-8")

created = []

try:
    # Recheck all source hashes immediately before writing.
    assert {
        p.relative_to(ROOT).as_posix(): sha256(p)
        for p in SOURCES
    } == source_hashes

    for path in OUTPUTS:
        with path.open("xb") as f:
            f.write(output_bytes[path])
        created.append(path)

    # Verify complete output after writing.
    for path, original_bytes in output_bytes.items():
        assert sha256(path) == hashlib.sha256(
            original_bytes
        ).hexdigest()

    assert len(read_csv(DATA_OUT)) == 3
    assert len(read_csv(MODEL_OUT)) == 2
    assert len(read_csv(ERROR_OUT)) == 6
    assert len(read_csv(TEMP_OUT)) == 3

    assert read_json(AUDIT_OUT)["status"] == (
        "RESULT_PART2_CONSOLIDATED_RESULTS_PASS"
    )

    assert {
        p.relative_to(ROOT).as_posix(): sha256(p)
        for p in SOURCES
    } == source_hashes

except Exception:
    # Roll back only files created by this execution.
    for path in created:
        if path.exists():
            path.unlink()
    raise

# ------------------------------------------------------------
# I. FINAL RESULT
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("06_RESULT PART2 — FINAL AUDIT")
print("=" * 75)

print("Dataset rows          : 348")
print("Leakage groups        :", unique_groups)
print("Models                :", len(model_table))
print("Per-class result rows :", len(class_table))
print("Disaster cases        :", len(temp_table))
print("Daily calendar rows   :", len(daily))
print("Source modified       : NO")
print("Model training        : NO")

print("\nMODEL EVALUATION:")
display(model_view.round(4))

print("\nPER-CLASS ERROR ANALYSIS:")
display(error_view.round(4))

print("\nTEMPORAL FINDINGS:")
display(temp_view)

print("\nOUTPUT FILES:")
for path in OUTPUTS:
    print("-", path.name)

print("\nFINAL STATUS:", audit["status"])
print("=" * 75)


RESULT PART2 — CONSOLIDATION PREFLIGHT
PART1 AUDIT & SOURCE SHA256: PASS
348 TWEET-LEVEL PREDICTIONS: PASS
DATASET CHARACTERISTICS: PASS
OOF METRICS & NEGATIVE CLASS RECOMPUTATION: PASS
PAIRED OOF ERROR CATEGORY COUNTS: PASS
TEMPORAL FINDINGS RECONCILIATION: PASS

06_RESULT PART2 — FINAL AUDIT
Dataset rows          : 348
Leakage groups        : 337
Models                : 2
Per-class result rows : 6
Disaster cases        : 3
Daily calendar rows   : 20
Source modified       : NO
Model training        : NO

MODEL EVALUATION:


,model,n_oof,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1
0,baseline,348,0.7040,0.7761,0.4943,0.5236,0.6898
1,lstm,348,0.5862,0.4760,0.5596,0.4751,0.6175



PER-CLASS ERROR ANALYSIS:


,model,sentiment_label,support,true_positive,false_positive,false_negative,precision,recall,f1
0,baseline,NETRAL,225,187,62,38,0.7510,0.8311,0.7890
1,baseline,POSITIF,108,56,41,52,0.5773,0.5185,0.5463
2,baseline,NEGATIF,15,2,0,13,1.0000,0.1333,0.2353
3,lstm,NETRAL,225,139,36,86,0.7943,0.6178,0.6950
4,lstm,POSITIF,108,57,58,51,0.4957,0.5278,0.5112
5,lstm,NEGATIF,15,8,50,7,0.1379,0.5333,0.2192



TEMPORAL FINDINGS:


,case_name,calendar_days,observed_days,zero_tweet_days,peak_date_wib,peak_daily_tweets
0,Lombok 2018,9,6,3,2018-08-10,24
1,Palu 2018,5,5,0,2018-10-01,36
2,Sumatera 2025,6,4,2,2025-12-03,54



OUTPUT FILES:
- dataset_characteristics.csv
- consolidated_model_evaluation.csv
- model_error_analysis_by_class.csv
- consolidated_temporal_findings.csv
- consolidated_research_results.md
- part2_consolidated_results_audit.json

FINAL STATUS: RESULT_PART2_CONSOLIDATED_RESULTS_PASS


In [4]:
# 06_RESULT — PART3 / CELL 03
# RESULTS VISUALIZATION & FIGURE SELECTION
#
# 17 LOCKED FIGURES -> 10 MAIN + 7 APPENDIX
# ACADEMIC CAPTIONS | IMAGE PREVIEW LINKS
# SHA256 AUDIT | READ-ONLY SOURCE | NO OVERWRITE
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import quote
import hashlib
import json
import os
import uuid

import pandas as pd

ROOT = Path(r"C:\Penelitian Sentimen Bencana Nasional")

P1 = ROOT / "06_Result" / "outputs" / "cross_phase_integrity"
P2 = ROOT / "06_Result" / "outputs" / "consolidated_results"
OUT = ROOT / "06_Result" / "outputs" / "figure_selection"

REGISTRY = P1 / "integrated_figure_registry_17.csv"
P1_AUDIT = P1 / "part1_cross_phase_integrity_audit.json"
P2_AUDIT = P2 / "part2_consolidated_results_audit.json"
MODEL_METRICS = P2 / "consolidated_model_evaluation.csv"
TEMP_FINDINGS = P2 / "consolidated_temporal_findings.csv"
ERROR_TABLE = P2 / "model_error_analysis_by_class.csv"

SELECTION_OUT = OUT / "result_figure_selection_17.csv"
CAPTIONS_OUT = OUT / "result_academic_figure_captions.md"
PLAN_OUT = OUT / "result_bab4_figure_plan.md"
AUDIT_OUT = OUT / "part3_figure_selection_audit.json"

OUTPUTS = [
    SELECTION_OUT,
    CAPTIONS_OUT,
    PLAN_OUT,
    AUDIT_OUT
]

SOURCES = [
    REGISTRY, P1_AUDIT, P2_AUDIT,
    MODEL_METRICS, TEMP_FINDINGS, ERROR_TABLE
]

EXPECTED = {
    "MODEL_G01", "MODEL_G02", "MODEL_G03",
    "MODEL_G04", "MODEL_G05", "MODEL_G06",
    "MODEL_G07", "MODEL_G08", "MODEL_G09",
    "TEMP_T04_F01", "TEMP_T04_F02",
    "TEMP_T04_F03", "TEMP_T04_F04",
    "TEMP_T04_F05", "TEMP_T04_F06",
    "TEMP_T04_F07", "TEMP_T04_F08"
}

# Recommended main figure order.
MAIN = [
    "MODEL_G05",
    "MODEL_G04",
    "MODEL_G06",
    "TEMP_T04_F08",
    "TEMP_T04_F01",
    "TEMP_T04_F02",
    "TEMP_T04_F03",
    "TEMP_T04_F04",
    "TEMP_T04_F05",
    "TEMP_T04_F06"
]

# Supporting figures: not deleted.
APPENDIX = [
    "MODEL_G01",
    "MODEL_G02",
    "MODEL_G03",
    "MODEL_G07",
    "MODEL_G08",
    "MODEL_G09",
    "TEMP_T04_F07"
]

# Captions and research notes are carefully separated.
DESCRIPTIONS = {
    "MODEL_G05": (
        "Perbandingan metrik evaluasi out-of-fold antara "
        "baseline TF-IDF + Logistic Regression dan LSTM.",
        "4.3",
        "Baseline unggul pada OOF Macro-F1. "
        "LSTM memiliki Macro-Recall lebih tinggi."
    ),
    "MODEL_G04": (
        "Matriks kebingungan prediksi out-of-fold model LSTM "
        "pada tiga kelas sentimen.",
        "4.3",
        "Gunakan untuk membahas prediksi benar dan kesalahan "
        "antar kelas; bukan hasil independent test set."
    ),
    "MODEL_G06": (
        "Perbandingan F1-score per kelas sentimen antara "
        "baseline dan LSTM berdasarkan prediksi OOF.",
        "4.3",
        "Kelas NEGATIF memiliki support hanya 15 tweet; "
        "interpretasikan metrik kelas minoritas secara hati-hati."
    ),
    "TEMP_T04_F08": (
        "Komposisi label sentimen manusia pada sampel tweet "
        "Lombok 2018, Palu 2018, dan Sumatera 2025.",
        "4.6",
        "Menunjukkan proporsi sampel; bukan estimasi "
        "representatif opini masyarakat."
    ),
    "TEMP_T04_F01": (
        "Jumlah tweet menurut kelas sentimen per hari "
        "untuk kasus Lombok 2018 (WIB).",
        "4.5",
        "Tiga tanggal tanpa observasi memiliki jumlah nol; "
        "bukan bukti ketiadaan diskusi publik."
    ),
    "TEMP_T04_F02": (
        "Jumlah tweet menurut kelas sentimen per hari "
        "untuk kasus Palu 2018 (WIB).",
        "4.5",
        "Seluruh lima hari kalender memiliki observasi "
        "dalam sampel penelitian."
    ),
    "TEMP_T04_F03": (
        "Jumlah tweet menurut kelas sentimen per hari "
        "untuk kasus Sumatera 2025 (WIB).",
        "4.5",
        "Dua hari kalender tanpa observasi dipertahankan "
        "di dalam grafik."
    ),
    "TEMP_T04_F04": (
        "Persentase label sentimen harian pada sampel tweet "
        "Lombok 2018 beserta ukuran sampel harian.",
        "4.5",
        "Tanggal tanpa tweet memiliki persentase tidak "
        "terdefinisi dan tidak diinterpolasi."
    ),
    "TEMP_T04_F05": (
        "Persentase label sentimen harian pada sampel tweet "
        "Palu 2018 beserta ukuran sampel harian.",
        "4.5",
        "Perhatikan nilai n saat membandingkan "
        "perubahan persentase antartanggal."
    ),
    "TEMP_T04_F06": (
        "Persentase label sentimen harian pada sampel tweet "
        "Sumatera 2025 beserta ukuran sampel harian.",
        "4.5",
        "Hari tanpa observasi tidak ditampilkan "
        "sebagai persentase nol."
    ),
    "MODEL_G01": (
        "Kurva training loss model LSTM pada tiap fold.",
        "Lampiran",
        "Digunakan sebagai informasi diagnostik training."
    ),
    "MODEL_G02": (
        "Kurva training accuracy model LSTM pada tiap fold.",
        "Lampiran",
        "Akurasi training tidak sama dengan performa OOF."
    ),
    "MODEL_G03": (
        "Perbandingan Macro-F1 model LSTM menurut fold validasi.",
        "Lampiran",
        "Menunjukkan variasi performa antar fold."
    ),
    "MODEL_G07": (
        "Ukuran vocabulary LSTM pada masing-masing fold.",
        "Lampiran",
        "Menjelaskan karakteristik representasi teks."
    ),
    "MODEL_G08": (
        "Distribusi jumlah token pada tweet penelitian.",
        "Lampiran",
        "Menjelaskan variasi panjang teks."
    ),
    "MODEL_G09": (
        "Distribusi jumlah token menurut kelas sentimen.",
        "Lampiran",
        "Analisis deskriptif panjang teks per kelas."
    ),
    "TEMP_T04_F07": (
        "Perbandingan volume tweet berdasarkan hari relatif "
        "terhadap awal observasi setiap kasus.",
        "Lampiran",
        "Hari relatif 0 bukan tanggal terjadinya bencana "
        "maupun keputusan kebijakan."
    )
}

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            digest.update(chunk)
    return digest.hexdigest()

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def rel_repo(path):
    return path.relative_to(ROOT).as_posix()

def md_link(path):
    # Links must be relative to the output Markdown folder.
    relative = Path(os.path.relpath(path, OUT)).as_posix()
    return quote(relative, safe="/.-_")

def write_json(path, data):
    path.write_text(
        json.dumps(
            data, indent=2, ensure_ascii=False,
            allow_nan=False
        ),
        encoding="utf-8"
    )

print("=" * 72)
print("RESULT PART3 — FIGURE SELECTION PREFLIGHT")
print("=" * 72)

# ------------------------------------------------------------
# A. SOURCE PREFLIGHT
# ------------------------------------------------------------

for path in SOURCES:
    assert path.is_file(), f"Source missing: {path}"

for path in OUTPUTS:
    if path.exists():
        raise FileExistsError(
            f"Part3 output exists; no overwrite: {path}"
        )

source_before = {
    rel_repo(path): sha256(path)
    for path in SOURCES
}

audit1 = read_json(P1_AUDIT)
audit2 = read_json(P2_AUDIT)

assert audit1["status"] == (
    "RESULT_PART1_CROSS_PHASE_INTEGRITY_PASS"
)
assert audit2["status"] == (
    "RESULT_PART2_CONSOLIDATED_RESULTS_PASS"
)

# Part1 must attest to the original figure registry.
assert (
    audit1["output_sha256"][REGISTRY.name]
    == sha256(REGISTRY)
)

# Part2 must attest to the consolidated tables.
for table_path in [MODEL_METRICS, TEMP_FINDINGS, ERROR_TABLE]:
    assert (
        audit2["output_sha256"][table_path.name]
        == sha256(table_path)
    )

assert audit1["dataset_rows"] == 348
assert audit1["total_figures_verified"] == 17
assert audit2["dataset_rows"] == 348
assert audit2["model_count"] == 2

print("RESULT PART1/PART2 SOURCE AUDITS: PASS")

# ------------------------------------------------------------
# B. VALIDATE CONSOLIDATED TABLES
# ------------------------------------------------------------

metrics = pd.read_csv(
    MODEL_METRICS, encoding="utf-8-sig"
)
temporal = pd.read_csv(
    TEMP_FINDINGS, encoding="utf-8-sig"
)
errors = pd.read_csv(
    ERROR_TABLE, encoding="utf-8-sig"
)

assert len(metrics) == 2
assert set(metrics["model"]) == {"baseline", "lstm"}
assert len(temporal) == 3
assert int(temporal["total_tweets"].sum()) == 348
assert len(errors) == 6
assert int(
    temporal["zero_tweet_days"].sum()
) == 5

m = metrics.set_index("model")
assert float(m.loc["baseline", "macro_f1"]) > (
    float(m.loc["lstm", "macro_f1"])
)
assert float(m.loc["lstm", "macro_recall"]) > (
    float(m.loc["baseline", "macro_recall"])
)

negative_rows = errors[
    errors["sentiment_label"] == "NEGATIF"
]
assert len(negative_rows) == 2
assert set(negative_rows["support"]) == {15}

print("CONSOLIDATED MODEL/TEMPORAL TABLES: PASS")

# ------------------------------------------------------------
# C. SELECT AND VERIFY ALL 17 FIGURES
# ------------------------------------------------------------

registry = pd.read_csv(
    REGISTRY, encoding="utf-8-sig"
)

assert len(registry) == 17
assert registry["integrated_figure_id"].is_unique
assert set(registry["integrated_figure_id"]) == EXPECTED
assert set(MAIN).isdisjoint(APPENDIX)
assert set(MAIN + APPENDIX) == EXPECTED
assert set(DESCRIPTIONS) == EXPECTED

by_id = registry.set_index("integrated_figure_id")

figure_sources = {}
figure_hashes = {}
selected_rows = []

for rank, fid in enumerate(MAIN + APPENDIX, start=1):
    record = by_id.loc[fid]
    source = ROOT / str(record["relative_repository_path"])

    assert source.is_file(), (
        f"Missing original figure: {source}"
    )
    assert source.resolve().is_relative_to(ROOT.resolve())

    with source.open("rb") as f:
        assert f.read(8) == bytes.fromhex(
            "89504e470d0a1a0a"
        )

    actual_hash = sha256(source)
    assert actual_hash == str(record["sha256"])
    assert actual_hash == audit1["figure_sha256"][
        rel_repo(source)
    ]

    figure_sources[fid] = source
    figure_hashes[rel_repo(source)] = actual_hash

    is_main = fid in MAIN
    seq = MAIN.index(fid) + 1 if is_main else APPENDIX.index(fid) + 1

    caption, section, note = DESCRIPTIONS[fid]

    selected_rows.append({
        "figure_id": fid,
        "stage": str(record["stage"]),
        "selection": "MAIN" if is_main else "APPENDIX",
        "sequence": int(seq),
        "provisional_number": (
            f"Gambar 4.{seq}" if is_main
            else f"Lampiran Gambar A.{seq}"
        ),
        "suggested_chapter_section": section,
        "academic_caption": caption,
        "interpretation_note": note,
        "original_repository_path": rel_repo(source),
        "markdown_relative_link": md_link(source),
        "source_sha256": actual_hash,
        "figure_size_bytes": source.stat().st_size
    })

selection = pd.DataFrame(selected_rows)

assert len(selection) == 17
assert (selection["selection"] == "MAIN").sum() == 10
assert (selection["selection"] == "APPENDIX").sum() == 7
assert selection["figure_id"].is_unique
assert selection["source_sha256"].nunique() == 17

print("17 ORIGINAL FIGURES & SHA256: PASS")
print("10 MAIN / 7 APPENDIX CLASSIFICATION: PASS")

# ------------------------------------------------------------
# D. ACADEMIC CAPTIONS WITH ORIGINAL IMAGE PREVIEWS
# ------------------------------------------------------------

caption_lines = [
    "# DAFTAR GAMBAR HASIL PENELITIAN",
    "",
    "**Fase:** 06_Result Part3",
    "",
    "**Catatan:** Penomoran bersifat sementara dan akan "
    "disesuaikan dengan posisi gambar di BAB IV.",
    "",
    "## A. Grafik Utama BAB IV",
    ""
]

for row in selection[
    selection["selection"] == "MAIN"
].itertuples(index=False):
    caption_lines.extend([
        f"### {row.provisional_number}",
        "",
        f"**Caption:** {row.academic_caption}",
        "",
        f"**Bagian:** {row.suggested_chapter_section}",
        "",
        f"![{row.figure_id}]({row.markdown_relative_link})",
        "",
        f"**Catatan interpretasi:** {row.interpretation_note}",
        "",
        f"**Sumber:** `{row.original_repository_path}`",
        ""
    ])

caption_lines.extend([
    "## B. Grafik Pendukung / Lampiran",
    ""
])

for row in selection[
    selection["selection"] == "APPENDIX"
].itertuples(index=False):
    caption_lines.extend([
        f"### {row.provisional_number}",
        "",
        f"**Caption:** {row.academic_caption}",
        "",
        f"![{row.figure_id}]({row.markdown_relative_link})",
        "",
        f"**Catatan:** {row.interpretation_note}",
        ""
    ])

captions_md = "\n".join(caption_lines) + "\n"

# ------------------------------------------------------------
# E. FIGURE PLAN FOR BAB IV
# ------------------------------------------------------------

plan_md = """# RENCANA VISUALISASI BAB IV

**Penelitian:** Sentimen Penetapan Bencana Nasional

**Tahap:** 06_Result Part3

## 1. Prinsip Seleksi

Pemilihan grafik mengutamakan kaitan langsung dengan
rumusan masalah dan hasil penelitian utama.

Seluruh grafik berasal dari artefak Modeling dan Temporal
yang telah diaudit. Tidak ada grafik dilatih, dihitung ulang,
atau dibuat ulang dalam Part3.

Sepuluh gambar direkomendasikan sebagai visual utama.
Tujuh gambar menjadi pendukung atau lampiran.

## 2. Urutan Naratif yang Disarankan

### 4.3 Hasil Pemodelan

- Perbandingan metrik baseline vs LSTM
- Confusion matrix LSTM
- F1-score per kelas

Gunakan metrik OOF dan beri penjelasan
bahwa tidak ada independent holdout test set.

### 4.5 Analisis Temporal

- Jumlah tweet harian Lombok, Palu, Sumatera
- Persentase sentimen harian Lombok, Palu, Sumatera

Gunakan label manusia, bukan prediksi LSTM.
Semua tanggal dalam WIB.
Tanggal tanpa observasi bukan bukti aktivitas publik nol.
Pembahasan persentase harus memperhatikan ukuran sampel.

### 4.6 Perbandingan Kasus

- Komposisi sentimen pada tiga kasus

Grafik menggambarkan proporsi sampel tweet,
bukan estimasi opini seluruh masyarakat.

## 3. Lampiran

- Training loss per fold
- Training accuracy per fold
- Macro-F1 LSTM per fold
- Vocabulary size per fold
- Token count distribution
- Token count by label
- Relative-day tweet volume

Untuk grafik relative-day, hari 0 adalah hari pertama
observasi, bukan awal kejadian bencana.

## 4. Batasan Interpretasi

- Sampel berlabel berjumlah 348 tweet.
- NEGATIF hanya memiliki 15 tweet aktual.
- Evaluasi model memakai SGKF 3-fold OOF.
- Grafik temporal memakai label manusia yang dikunci.
- Periode pengamatan antarkasus berbeda.
- Tidak dilakukan inferensi populasi atau kausal.
- Penomoran Gambar 4.x bersifat provisional.

## 5. Ketentuan Reproducibility

Seluruh PNG direferensikan ke lokasi aslinya.
Tidak ada penggandaan atau perubahan file gambar.
SHA-256 disimpan di result_figure_selection_17.csv
dan part3_figure_selection_audit.json.

## 6. Pemeriksaan Editorial

Buka Markdown preview sebelum finalisasi.
Periksa keterbacaan label, legenda, caption,
dan posisi gambar pada dokumen akhir.

**Tahap berikutnya:** Result Part4 —
Discussion & Research Question Alignment.
"""

# ------------------------------------------------------------
# F. AUDIT AND IMMUTABLE EXPORT
# ------------------------------------------------------------

generated_at = datetime.now(timezone.utc).isoformat()

audit3 = {
    "stage": "06_Result",
    "part": "3 Results Visualization and Figure Selection",
    "generated_at_utc": generated_at,
    "source_commit":
        "e4b8124a413f01763497ca8d098d1af3412f71db",
    "source_sha256": source_before,
    "original_figure_sha256": figure_hashes,
    "original_figures_verified": 17,
    "main_figures": 10,
    "appendix_figures": 7,
    "source_figures_modified": False,
    "new_png_generated": False,
    "model_training_performed": False,
    "provisional_figure_numbering": True,
    "figure_selection_complete": True,
    "visual_layout_manual_review_required": True,
    "output_sha256": {},
    "status": "RESULT_PART3_FIGURE_SELECTION_PASS"
}

OUT.mkdir(parents=True, exist_ok=True)

for output in OUTPUTS:
    if output.exists():
        raise FileExistsError(
            f"Refusing overwrite: {output}"
        )

token = uuid.uuid4().hex[:10]

temps = {
    path: OUT / f".result_part3_{token}_{i}.tmp"
    for i, path in enumerate(OUTPUTS)
}

created = []

try:
    selection.to_csv(
        temps[SELECTION_OUT],
        index=False, encoding="utf-8-sig"
    )

    temps[CAPTIONS_OUT].write_text(
        captions_md, encoding="utf-8"
    )

    temps[PLAN_OUT].write_text(
        plan_md, encoding="utf-8"
    )

    audit3["output_sha256"] = {
        output.name: sha256(temps[output])
        for output in OUTPUTS[:-1]
    }

    write_json(temps[AUDIT_OUT], audit3)

    # Verify generated output before publication.
    saved_selection = pd.read_csv(
        temps[SELECTION_OUT], encoding="utf-8-sig"
    )

    assert len(saved_selection) == 17
    assert (
        saved_selection["selection"] == "MAIN"
    ).sum() == 10
    assert (
        saved_selection["selection"] == "APPENDIX"
    ).sum() == 7

    for row in selection.itertuples(index=False):
        assert (
            f"]({row.markdown_relative_link})"
            in captions_md
        )
        assert figure_sources[row.figure_id].is_file()

    # All sources must remain unchanged.
    assert {
        rel_repo(p): sha256(p)
        for p in SOURCES
    } == source_before

    assert {
        rel_repo(p): sha256(p)
        for p in figure_sources.values()
    } == figure_hashes

    # Publish without replacing any existing artifact.
    for destination in OUTPUTS:
        with destination.open("xb") as f:
            f.write(temps[destination].read_bytes())
        created.append(destination)

except Exception:
    for destination in created:
        if destination.exists():
            destination.unlink()
    raise

finally:
    for temporary in temps.values():
        if temporary.exists():
            temporary.unlink()

# ------------------------------------------------------------
# G. FINAL VERIFICATION
# ------------------------------------------------------------

for filename, expected_hash in (
    audit3["output_sha256"].items()
):
    assert sha256(OUT / filename) == expected_hash

assert {
    rel_repo(p): sha256(p)
    for p in SOURCES
} == source_before

assert {
    rel_repo(p): sha256(p)
    for p in figure_sources.values()
} == figure_hashes

assert read_json(AUDIT_OUT)["status"] == (
    "RESULT_PART3_FIGURE_SELECTION_PASS"
)

print("\n" + "=" * 72)
print("RESULT PART3 — FINAL AUDIT")
print("=" * 72)
print("Original figures :", len(selection))
print("Main figures     :", 10)
print("Appendix figures :", 7)
print("New PNG created  : NO")
print("Sources modified : NO")
print("Model training   : NO")

print("\nSELECTED MAIN FIGURES")
display(
    selection[
        selection["selection"] == "MAIN"
    ][[
        "provisional_number",
        "figure_id",
        "suggested_chapter_section",
        "academic_caption"
    ]]
)

print("\nOUTPUT FILES")
for path in OUTPUTS:
    print("-", path)

print("\nFINAL STATUS:", audit3["status"])
print("=" * 72)

RESULT PART3 — FIGURE SELECTION PREFLIGHT
RESULT PART1/PART2 SOURCE AUDITS: PASS
CONSOLIDATED MODEL/TEMPORAL TABLES: PASS
17 ORIGINAL FIGURES & SHA256: PASS
10 MAIN / 7 APPENDIX CLASSIFICATION: PASS

RESULT PART3 — FINAL AUDIT
Original figures : 17
Main figures     : 10
Appendix figures : 7
New PNG created  : NO
Sources modified : NO
Model training   : NO

SELECTED MAIN FIGURES


,provisional_number,figure_id,suggested_chapter_section,academic_caption
0,Gambar 4.1,MODEL_G05,4.3,Perbandingan metrik evaluasi out-of-fold antar...
1,Gambar 4.2,MODEL_G04,4.3,Matriks kebingungan prediksi out-of-fold model...
2,Gambar 4.3,MODEL_G06,4.3,Perbandingan F1-score per kelas sentimen antar...
3,Gambar 4.4,TEMP_T04_F08,4.6,Komposisi label sentimen manusia pada sampel t...
4,Gambar 4.5,TEMP_T04_F01,4.5,Jumlah tweet menurut kelas sentimen per hari u...
5,Gambar 4.6,TEMP_T04_F02,4.5,Jumlah tweet menurut kelas sentimen per hari u...
6,Gambar 4.7,TEMP_T04_F03,4.5,Jumlah tweet menurut kelas sentimen per hari u...
7,Gambar 4.8,TEMP_T04_F04,4.5,Persentase label sentimen harian pada sampel t...
8,Gambar 4.9,TEMP_T04_F05,4.5,Persentase label sentimen harian pada sampel t...
9,Gambar 4.10,TEMP_T04_F06,4.5,Persentase label sentimen harian pada sampel t...



OUTPUT FILES
- C:\Penelitian Sentimen Bencana Nasional\06_Result\outputs\figure_selection\result_figure_selection_17.csv
- C:\Penelitian Sentimen Bencana Nasional\06_Result\outputs\figure_selection\result_academic_figure_captions.md
- C:\Penelitian Sentimen Bencana Nasional\06_Result\outputs\figure_selection\result_bab4_figure_plan.md
- C:\Penelitian Sentimen Bencana Nasional\06_Result\outputs\figure_selection\part3_figure_selection_audit.json

FINAL STATUS: RESULT_PART3_FIGURE_SELECTION_PASS


In [5]:
# 06_RESULT — PART4 / CELL 04
# DISCUSSION & RESEARCH QUESTION ALIGNMENT
#
# Read-only sources | No retraining | No overwrite
# Proposal alignment | Methodological warnings | SHA256
# ==========================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import numpy as np
import pandas as pd

ROOT = Path(r"C:\Penelitian Sentimen Bencana Nasional")
BASE = ROOT / "06_Result" / "outputs"
P1 = BASE / "cross_phase_integrity"
P2 = BASE / "consolidated_results"
P3 = BASE / "figure_selection"
OUT = BASE / "discussion_alignment"

MODEL = P2 / "consolidated_model_evaluation.csv"
ERRORS = P2 / "model_error_analysis_by_class.csv"
DATA = P2 / "dataset_characteristics.csv"
TEMP = P2 / "consolidated_temporal_findings.csv"
AUDIT2 = P2 / "part2_consolidated_results_audit.json"
FIGURES = P3 / "result_figure_selection_17.csv"
AUDIT3 = P3 / "part3_figure_selection_audit.json"

ALIGN_OUT = OUT / "research_question_alignment.csv"
DISCUSSION_OUT = OUT / "research_discussion_draft.md"
ISSUES_OUT = OUT / "methodological_alignment_issues.json"
AUDIT_OUT = OUT / "part4_discussion_alignment_audit.json"

SOURCES = [
    MODEL, ERRORS, DATA, TEMP,
    AUDIT2, FIGURES, AUDIT3
]

OUTPUTS = [
    ALIGN_OUT, DISCUSSION_OUT,
    ISSUES_OUT, AUDIT_OUT
]

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def write_json_bytes(value):
    return json.dumps(
        value, ensure_ascii=False,
        indent=2, allow_nan=False
    ).encode("utf-8")

def relative(path):
    return path.relative_to(ROOT).as_posix()

print("=" * 72)
print("RESULT PART4 — DISCUSSION ALIGNMENT PREFLIGHT")
print("=" * 72)

# ----------------------------------------------------------
# A. LOCKED SOURCES
# ----------------------------------------------------------

for path in SOURCES:
    assert path.is_file(), f"Missing source: {path}"

for path in OUTPUTS:
    if path.exists():
        raise FileExistsError(
            f"Existing output, refusing overwrite: {path}"
        )

source_before = {
    relative(path): sha256(path)
    for path in SOURCES
}

audit2 = read_json(AUDIT2)
audit3 = read_json(AUDIT3)

assert audit2["status"] == (
    "RESULT_PART2_CONSOLIDATED_RESULTS_PASS"
)
assert audit3["status"] == (
    "RESULT_PART3_FIGURE_SELECTION_PASS"
)

for path in [MODEL, ERRORS, DATA, TEMP]:
    assert sha256(path) == (
        audit2["output_sha256"][path.name]
    )

assert sha256(FIGURES) == (
    audit3["output_sha256"][FIGURES.name]
)

print("PART2/PART3 AUDIT & SOURCE HASHES: PASS")

# ----------------------------------------------------------
# B. LOAD AND VERIFY RESEARCH RESULTS
# ----------------------------------------------------------

model = pd.read_csv(MODEL, encoding="utf-8-sig")
errors = pd.read_csv(ERRORS, encoding="utf-8-sig")
dataset = pd.read_csv(DATA, encoding="utf-8-sig")
temporal = pd.read_csv(TEMP, encoding="utf-8-sig")
figures = pd.read_csv(FIGURES, encoding="utf-8-sig")

assert len(model) == 2
assert len(errors) == 6
assert len(dataset) == 3
assert len(temporal) == 3
assert len(figures) == 17

assert int(dataset["total_tweets"].sum()) == 348
assert int(temporal["total_tweets"].sum()) == 348
assert int(temporal["calendar_days"].sum()) == 20
assert int(temporal["zero_tweet_days"].sum()) == 5

assert set(model["model"]) == {"baseline", "lstm"}
assert (
    figures["selection"].value_counts().to_dict()
    == {"MAIN": 10, "APPENDIX": 7}
)

m = model.set_index("model")
b = m.loc["baseline"]
l = m.loc["lstm"]

assert b["macro_f1"] > l["macro_f1"]
assert l["macro_recall"] > b["macro_recall"]

neg = errors[
    errors["sentiment_label"] == "NEGATIF"
].set_index("model")

assert int(neg.loc["baseline", "true_positive"]) == 2
assert int(neg.loc["lstm", "true_positive"]) == 8
assert int(neg.loc["lstm", "false_positive"]) == 50

total_labels = {
    label: int(dataset[f"count_{label}"].sum())
    for label in ["NETRAL", "POSITIF", "NEGATIF"]
}

assert total_labels == {
    "NETRAL": 225,
    "POSITIF": 108,
    "NEGATIF": 15
}

print("CONSOLIDATED RESEARCH RESULTS: PASS")

# ----------------------------------------------------------
# C. PROPOSAL-BASED QUESTION ALIGNMENT
# ----------------------------------------------------------

# These questions/objectives are transcribed from sections
# 1.2 and 1.4 of G1F020005_Mariana ClaudyaNEW.pdf.
#
# The original PDF is not a runtime-locked input here.
# Therefore the audit will explicitly distinguish
# proposal text transcription from verified CSV sources.

alignment_rows = [
    {
        "research_question_id": "RM1",
        "research_question":
            "Bagaimana penerapan metode Long Short-Term "
            "Memory (LSTM) dalam menganalisis sentimen "
            "masyarakat terhadap kebijakan penanggulangan "
            "bencana alam di Indonesia di Aplikasi X?",
        "research_objective":
            "Menerapkan LSTM dalam analisis sentimen "
            "komentar masyarakat dari X (Twitter).",
        "evidence":
            "Modeling Part20; LSTM OOF metrics; "
            "training/evaluation figures",
        "discussion_section": "4.3",
        "evidence_status": "SUPPORTED_WITH_LIMITATIONS",
        "limitation":
            "Kinerja LSTM dievaluasi dengan OOF; "
            "tidak ada independent holdout test set."
    },
    {
        "research_question_id": "RM2",
        "research_question":
            "Bagaimana hasil klasifikasi sentimen "
            "masyarakat yang terbagi ke dalam kategori "
            "positif, negatif, dan netral menggunakan "
            "metode LSTM?",
        "research_objective":
            "Mengklasifikasikan komentar ke dalam "
            "POSITIF, NEGATIF, dan NETRAL menggunakan LSTM.",
        "evidence":
            "OOF predictions, per-class precision, "
            "recall, F1, and LSTM confusion matrix",
        "discussion_section": "4.3–4.4",
        "evidence_status": "SUPPORTED_WITH_LIMITATIONS",
        "limitation":
            "NEGATIF support=15; LSTM false-positive "
            "NEGATIF=50; interpretasi dibatasi sampel."
    },
    {
        "research_question_id": "RM3",
        "research_question":
            "Bagaimana pola dinamika temporal sentimen "
            "masyarakat terhadap kebijakan "
            "penanggulangan bencana dari waktu ke waktu?",
        "research_objective":
            "Menganalisis dinamika temporal sentimen "
            "masyarakat dari waktu ke waktu.",
        "evidence":
            "Temporal Part3–5; daily human-label counts "
            "and percentages; 8 temporal figures",
        "discussion_section": "4.5–4.6",
        "evidence_status": "DESCRIPTIVE_HUMAN_LABEL_ONLY",
        "limitation":
            "Agregasi temporal menggunakan label manusia, "
            "bukan prediksi LSTM; tidak ada analisis "
            "kausal atau estimasi populasi."
    }
]

alignment = pd.DataFrame(alignment_rows)

assert len(alignment) == 3
assert alignment["research_question_id"].is_unique
assert set(alignment["research_question_id"]) == {
    "RM1", "RM2", "RM3"
}

# ----------------------------------------------------------
# D. METHODOLOGICAL ALIGNMENT ISSUES
# ----------------------------------------------------------

issues = {
    "stage": "06_Result",
    "part": "4 Discussion & Research Question Alignment",
    "proposal_reference": "G1F020005_Mariana ClaudyaNEW.pdf",
    "proposal_text_source":
        "Transcription of sections 1.2, 1.3, 1.4; "
        "original PDF not hash-locked in this cell",
    "issues": [
        {
            "id": "METHOD_01",
            "severity": "HIGH",
            "topic": "Baseline comparison outside proposal boundary",
            "proposal_position":
                "Section 1.3 states LSTM without comparison "
                "to another method.",
            "implementation":
                "TF-IDF + Logistic Regression benchmark "
                "is evaluated alongside LSTM.",
            "required_action":
                "Seek supervisor approval to revise the "
                "methodological boundary or explicitly "
                "document the benchmark as an added evaluation.",
            "resolved": False
        },
        {
            "id": "METHOD_02",
            "severity": "HIGH",
            "topic": "Temporal labels versus LSTM predictions",
            "proposal_position":
                "Research objectives emphasize LSTM "
                "sentiment classification and temporal analysis.",
            "implementation":
                "Temporal timeline currently aggregates "
                "locked human sentiment labels.",
            "required_action":
                "Explicitly distinguish annotation-based "
                "temporal analysis from model-based "
                "classification; confirm intended scope "
                "with supervisor.",
            "resolved": False
        },
        {
            "id": "METHOD_03",
            "severity": "MEDIUM",
            "topic": "Policy terminology and causal interpretation",
            "proposal_position":
                "Proposal alternates between national disaster "
                "designation and disaster management policy.",
            "implementation":
                "Observed tweet labels do not establish "
                "effects of a particular policy decision.",
            "required_action":
                "Standardize operational definitions and "
                "avoid causal/policy-impact claims.",
            "resolved": False
        },
        {
            "id": "METHOD_04",
            "severity": "MEDIUM",
            "topic": "Sample coverage and test design",
            "proposal_position":
                "Objectives concern public sentiment over time.",
            "implementation":
                "348 collected tweets, 15 NEGATIF, "
                "3-fold OOF without independent holdout.",
            "required_action":
                "Report class imbalance, sampling limits "
                "and OOF generalization uncertainty.",
            "resolved": False
        }
    ],
    "status": "SUPERVISOR_ALIGNMENT_REQUIRED"
}

# ----------------------------------------------------------
# E. CREATE ACADEMIC DISCUSSION DRAFT
# ----------------------------------------------------------

def pct(value):
    return f"{value * 100:.2f}%"

case_sentences = []

for row in temporal.itertuples(index=False):
    case_sentences.append(
        f"- **{row.case_name}:** {row.total_tweets} tweet, "
        f"{row.observed_days}/{row.calendar_days} hari "
        f"berobservasi, {row.zero_tweet_days} hari tanpa "
        f"observasi. Puncak sampel harian "
        f"{row.peak_daily_tweets} tweet pada "
        f"{row.peak_date_wib} (WIB)."
    )

discussion = f"""# DRAFT PEMBAHASAN DAN KESESUAIAN RUMUSAN MASALAH

**Penelitian:** Sentimen Penetapan Bencana Nasional

**Fase:** 06_Result Part4

**Sifat dokumen:** Draft akademik, bukan naskah
final yang telah disetujui pembimbing.

## 1. Kerangka Pembahasan

Rumusan masalah mengacu pada bagian 1.2 proposal
`G1F020005_Mariana ClaudyaNEW.pdf`.

Penelitian menggunakan 348 tweet berlabel manusia,
dengan distribusi NETRAL 225, POSITIF 108,
dan NEGATIF 15 tweet.

Hasil Modeling dan Temporal mengacu pada 348 tweet
yang sama, sebagaimana diaudit pada Result Part1.

## 2. Pembahasan RM1 — Penerapan LSTM

Model LSTM diterapkan pada data tweet berbahasa
Indonesia dengan evaluasi Stratified Group K-Fold
tiga fold dan prediksi out-of-fold.

LSTM menghasilkan accuracy **{pct(l["accuracy"])}**
dan Macro-F1 **{l["macro_f1"]:.4f}**.

Sebagai benchmark tambahan, baseline
TF-IDF + Logistic Regression memperoleh accuracy
**{pct(b["accuracy"])}** dan Macro-F1
**{b["macro_f1"]:.4f}**.

Selisih Macro-F1 baseline terhadap LSTM
sebesar **{b["macro_f1"] - l["macro_f1"]:.4f}**.
Macro-Recall LSTM mencapai
**{l["macro_recall"]:.4f}**, lebih tinggi dibandingkan
baseline **{b["macro_recall"]:.4f}**.

Hasil ini tidak membuktikan bahwa LSTM unggul
dalam seluruh indikator. Evaluasi memakai OOF,
bukan independent holdout.

**Catatan penyelarasan:** proposal awal menyatakan
LSTM tanpa perbandingan metode. Penggunaan baseline
harus dikonfirmasi dan dijelaskan bersama pembimbing.

## 3. Pembahasan RM2 — Hasil Klasifikasi LSTM

Klasifikasi mencakup tiga kategori sentimen:
NETRAL, POSITIF, dan NEGATIF.

Pada kelas NEGATIF, hanya terdapat 15 tweet
aktual. LSTM memperoleh true positive
**{int(neg.loc["lstm", "true_positive"])}**,
false negative
**{int(neg.loc["lstm", "false_negative"])}**
dan false positive
**{int(neg.loc["lstm", "false_positive"])}**.

Recall NEGATIF LSTM sebesar
**{pct(neg.loc["lstm", "recall"])}**,
tetapi precision hanya
**{pct(neg.loc["lstm", "precision"])}**.

Dengan demikian, kenaikan sensitivitas model
terhadap NEGATIF disertai peningkatan prediksi
NEGATIF yang keliru.

Kinerja tidak dapat dinilai berdasarkan accuracy
saja karena ketidakseimbangan kelas.

Hasil klasifikasi LSTM adalah prediksi model yang
dievaluasi terhadap label manusia, bukan pengukuran
langsung seluruh opini publik.

## 4. Pembahasan RM3 — Dinamika Temporal

Analisis Temporal menggunakan timestamp publikasi
tweet yang dikonversi menjadi WIB dan label
manusia yang telah dikunci.

Total cakupan adalah 20 hari kalender kasus,
terdiri atas 15 hari dengan observasi dan
5 hari tanpa tweet pada dataset.

{chr(10).join(case_sentences)}

NETRAL menjadi kategori terbesar pada ketiga
kasus. Fluktuasi jumlah dan proporsi harian
merupakan gambaran sampel tweet yang terkumpul.

Tanggal tanpa tweet tidak boleh diartikan
sebagai tidak adanya percakapan publik.

Hari relatif dalam grafik adalah hari sejak awal
jendela observasi, bukan hari sejak kejadian
bencana atau keputusan penetapan status.

**Batasan penting:** hasil temporal saat ini
menggunakan label manusia, bukan hasil prediksi
LSTM. Karena itu, hasil ini menjawab dinamika
label sentimen dalam sampel, belum membuktikan
dinamika prediksi berbasis model LSTM.

## 5. Integrasi Ketiga Rumusan Masalah

RM1 dijawab melalui implementasi dan evaluasi LSTM.

RM2 dijawab melalui performa prediksi LSTM
pada tiga kategori sentimen dan analisis kesalahan.

RM3 dijawab secara deskriptif melalui agregasi
harian sentimen berlabel manusia untuk tiga kasus.

Interpretasi RM1–RM3 harus membedakan
evaluasi prediksi dari karakteristik data berlabel.

## 6. Keterbatasan dan Validitas

1. Dataset berisi 348 tweet, bukan sampel
   representatif seluruh pengguna X.
2. Kelas NEGATIF hanya 15 tweet.
3. Tidak tersedia independent holdout test set.
4. Periode pengamatan tiga kasus tidak sama.
5. Lima hari kalender kasus tidak memiliki observasi.
6. Tidak terdapat bukti hubungan sebab-akibat
   antara kebijakan dan pergeseran sentimen.
7. Benchmark baseline belum selaras dengan batasan
   metode pada proposal.
8. Agregasi temporal saat ini tidak menggunakan
   prediksi LSTM.

## 7. Keputusan Metodologis Sebelum BAB IV Final

- Konfirmasi kepada pembimbing apakah baseline
  Logistic Regression boleh dipertahankan sebagai
  benchmark tambahan.
- Konfirmasi apakah RM3 cukup dianalisis dari
  label manusia atau memerlukan analisis tambahan
  dari prediksi model.
- Seragamkan definisi penetapan bencana nasional
  dengan kebijakan penanggulangan bencana.
- Pertahankan penjelasan sampling, class imbalance,
  dan keterbatasan generalisasi.

**Status:** Draft pembahasan siap ditinjau.
Persetujuan metodologis masih diperlukan.
"""

# ----------------------------------------------------------
# F. IMMUTABLE EXPORT AND AUDIT
# ----------------------------------------------------------

generated_at = datetime.now(timezone.utc).isoformat()

audit4 = {
    "stage": "06_Result",
    "part": "4 Discussion & Research Question Alignment",
    "generated_at_utc": generated_at,
    "source_sha256": source_before,
    "proposal_file":
        "G1F020005_Mariana ClaudyaNEW.pdf",
    "proposal_pdf_sha256_verified": False,
    "research_questions_mapped": 3,
    "dataset_rows": 348,
    "model_count": 2,
    "temporal_cases": 3,
    "figure_registry_rows": 17,
    "methodological_issues": len(issues["issues"]),
    "unresolved_methodological_issues": len(issues["issues"]),
    "discussion_draft_prepared": True,
    "supervisor_review_required": True,
    "source_modified": False,
    "model_training_performed": False,
    "output_sha256": {},
    "status": "RESULT_PART4_DRAFT_ALIGNMENT_PASS"
}

output_data = {
    ALIGN_OUT: alignment.to_csv(
        index=False
    ).encode("utf-8-sig"),
    DISCUSSION_OUT: discussion.encode("utf-8"),
    ISSUES_OUT: write_json_bytes(issues)
}

audit4["output_sha256"] = {
    path.name: hashlib.sha256(blob).hexdigest()
    for path, blob in output_data.items()
}

output_data[AUDIT_OUT] = write_json_bytes(audit4)

OUT.mkdir(parents=True, exist_ok=True)

for path in OUTPUTS:
    if path.exists():
        raise FileExistsError(
            f"Refusing overwrite: {path}"
        )

created = []

try:
    assert {
        relative(path): sha256(path)
        for path in SOURCES
    } == source_before

    for destination in OUTPUTS:
        with destination.open("xb") as f:
            f.write(output_data[destination])
        created.append(destination)

    for destination, blob in output_data.items():
        assert sha256(destination) == (
            hashlib.sha256(blob).hexdigest()
        )

    assert len(
        pd.read_csv(ALIGN_OUT, encoding="utf-8-sig")
    ) == 3

    assert {
        relative(path): sha256(path)
        for path in SOURCES
    } == source_before

except Exception:
    for destination in created:
        if destination.exists():
            destination.unlink()
    raise

print("\n" + "=" * 72)
print("RESULT PART4 — FINAL AUDIT")
print("=" * 72)

print("Research questions mapped :", 3)
print("Dataset tweets             :", 348)
print("Models referenced          :", 2)
print("Temporal cases             :", 3)
print("Figure registry            :", 17)
print("Methodological issues      :", len(issues["issues"]))
print("Supervisor review required : YES")
print("Model training             : NO")
print("Source modification        : NO")

print("\nRESEARCH QUESTION ALIGNMENT")
display(alignment[[
    "research_question_id",
    "discussion_section",
    "evidence_status"
]])

print("\nOUTPUT FILES")
for path in OUTPUTS:
    print("-", path.name)

print("\nFINAL STATUS:", audit4["status"])
print("=" * 72)


RESULT PART4 — DISCUSSION ALIGNMENT PREFLIGHT
PART2/PART3 AUDIT & SOURCE HASHES: PASS
CONSOLIDATED RESEARCH RESULTS: PASS

RESULT PART4 — FINAL AUDIT
Research questions mapped : 3
Dataset tweets             : 348
Models referenced          : 2
Temporal cases             : 3
Figure registry            : 17
Methodological issues      : 4
Supervisor review required : YES
Model training             : NO
Source modification        : NO

RESEARCH QUESTION ALIGNMENT


,research_question_id,discussion_section,evidence_status
0,RM1,4.3,SUPPORTED_WITH_LIMITATIONS
1,RM2,4.3–4.4,SUPPORTED_WITH_LIMITATIONS
2,RM3,4.5–4.6,DESCRIPTIVE_HUMAN_LABEL_ONLY



OUTPUT FILES
- research_question_alignment.csv
- research_discussion_draft.md
- methodological_alignment_issues.json
- part4_discussion_alignment_audit.json

FINAL STATUS: RESULT_PART4_DRAFT_ALIGNMENT_PASS


In [6]:
# 06_RESULT — PART5 / CELL 05
# PROVISIONAL INTEGRATED RESEARCH REPORT
#
# Part1-4 locked inputs | RM1-RM3
# 10 original main figures | 4 issues remain OPEN
# NO TRAINING | NO OVERWRITE | SHA256 AUDIT
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import quote
import hashlib
import json
import os
import pandas as pd
import numpy as np

ROOT = Path(r"C:\Penelitian Sentimen Bencana Nasional")
BASE = ROOT / "06_Result" / "outputs"

P2 = BASE / "consolidated_results"
P3 = BASE / "figure_selection"
P4 = BASE / "discussion_alignment"
OUT = BASE / "integrated_report"

MODEL = P2 / "consolidated_model_evaluation.csv"
DATA = P2 / "dataset_characteristics.csv"
ERRORS = P2 / "model_error_analysis_by_class.csv"
TEMP = P2 / "consolidated_temporal_findings.csv"
AUDIT2 = P2 / "part2_consolidated_results_audit.json"

FIGURES = P3 / "result_figure_selection_17.csv"
AUDIT3 = P3 / "part3_figure_selection_audit.json"

QUESTIONS = P4 / "research_question_alignment.csv"
ISSUES = P4 / "methodological_alignment_issues.json"
DISCUSSION = P4 / "research_discussion_draft.md"
AUDIT4 = P4 / "part4_discussion_alignment_audit.json"

REPORT_OUT = OUT / "provisional_integrated_research_report.md"
ORDER_OUT = OUT / "final_main_figure_order_10.csv"
EVIDENCE_OUT = OUT / "research_question_evidence_matrix.csv"
ISSUES_OUT = OUT / "open_methodological_issues.json"
AUDIT_OUT = OUT / "part5_integrated_report_audit.json"

SOURCES = [
    MODEL, DATA, ERRORS, TEMP, AUDIT2,
    FIGURES, AUDIT3, QUESTIONS, ISSUES,
    DISCUSSION, AUDIT4
]

OUTPUTS = [
    REPORT_OUT, ORDER_OUT, EVIDENCE_OUT,
    ISSUES_OUT, AUDIT_OUT
]

EXPECTED_ISSUES = {
    "METHOD_01", "METHOD_02",
    "METHOD_03", "METHOD_04"
}

# New numbering follows the actual order of sections.
MAIN_ORDER = [
    "MODEL_G05",
    "MODEL_G04",
    "MODEL_G06",
    "TEMP_T04_F01",
    "TEMP_T04_F02",
    "TEMP_T04_F03",
    "TEMP_T04_F04",
    "TEMP_T04_F05",
    "TEMP_T04_F06",
    "TEMP_T04_F08"
]

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            digest.update(block)
    return digest.hexdigest()

def rel(path):
    return path.relative_to(ROOT).as_posix()

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def json_bytes(value):
    return json.dumps(
        value, ensure_ascii=False,
        indent=2, allow_nan=False
    ).encode("utf-8")

def fmt_num(x):
    if pd.isna(x):
        return ""
    if isinstance(x, (float, np.floating)):
        return f"{x:.4f}"
    return str(x).replace("|", "\\|").replace("\n", " ")

def md_table(df):
    headers = [str(x) for x in df.columns]
    rows = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |"
    ]
    for row in df.itertuples(index=False, name=None):
        rows.append(
            "| " + " | ".join(fmt_num(x) for x in row) + " |"
        )
    return "\n".join(rows)

def md_image_link(path):
    relative = Path(os.path.relpath(path, OUT)).as_posix()
    return quote(relative, safe="/.-_")

print("=" * 72)
print("06_RESULT PART5 — INTEGRATED REPORT PREFLIGHT")
print("=" * 72)

# ------------------------------------------------------------
# A. VERIFY ALL PREVIOUS AUDITS
# ------------------------------------------------------------

for p in SOURCES:
    assert p.is_file(), f"Missing source: {p}"

for p in OUTPUTS:
    if p.exists():
        raise FileExistsError(f"No overwrite: {p}")

a2 = read_json(AUDIT2)
a3 = read_json(AUDIT3)
a4 = read_json(AUDIT4)
issues_data = read_json(ISSUES)

assert a2["status"] == (
    "RESULT_PART2_CONSOLIDATED_RESULTS_PASS"
)
assert a3["status"] == (
    "RESULT_PART3_FIGURE_SELECTION_PASS"
)
assert a4["status"] == (
    "RESULT_PART4_DRAFT_ALIGNMENT_PASS"
)

# Trace every Part2/Part3/Part4 artifact to its audit.
for path in [MODEL, DATA, ERRORS, TEMP]:
    assert sha256(path) == a2["output_sha256"][path.name]

assert sha256(FIGURES) == (
    a3["output_sha256"][FIGURES.name]
)

for path in [QUESTIONS, ISSUES, DISCUSSION]:
    assert sha256(path) == a4["output_sha256"][path.name]

source_hashes = {rel(p): sha256(p) for p in SOURCES}

print("PART2 / PART3 / PART4 AUDIT CHAIN: PASS")

# ------------------------------------------------------------
# B. LOAD RESULTS
# ------------------------------------------------------------

models = pd.read_csv(MODEL, encoding="utf-8-sig")
dataset = pd.read_csv(DATA, encoding="utf-8-sig")
errors = pd.read_csv(ERRORS, encoding="utf-8-sig")
temporal = pd.read_csv(TEMP, encoding="utf-8-sig")
figures = pd.read_csv(FIGURES, encoding="utf-8-sig")
questions = pd.read_csv(QUESTIONS, encoding="utf-8-sig")

assert len(models) == 2
assert len(dataset) == 3
assert len(errors) == 6
assert len(temporal) == 3
assert len(figures) == 17
assert len(questions) == 3

assert int(dataset["total_tweets"].sum()) == 348
assert int(temporal["total_tweets"].sum()) == 348
assert int(temporal["calendar_days"].sum()) == 20
assert int(temporal["observed_days"].sum()) == 15
assert int(temporal["zero_tweet_days"].sum()) == 5

assert int(dataset["count_NETRAL"].sum()) == 225
assert int(dataset["count_POSITIF"].sum()) == 108
assert int(dataset["count_NEGATIF"].sum()) == 15

assert set(models["model"]) == {"baseline", "lstm"}

m = models.set_index("model")
b = m.loc["baseline"]
l = m.loc["lstm"]

assert b["macro_f1"] > l["macro_f1"]
assert l["macro_recall"] > b["macro_recall"]

neg = errors.loc[
    errors["sentiment_label"] == "NEGATIF"
].set_index("model")

assert int(neg.loc["baseline", "true_positive"]) == 2
assert int(neg.loc["baseline", "false_negative"]) == 13
assert int(neg.loc["lstm", "true_positive"]) == 8
assert int(neg.loc["lstm", "false_positive"]) == 50
assert int(neg.loc["lstm", "false_negative"]) == 7

assert set(questions["research_question_id"]) == {
    "RM1", "RM2", "RM3"
}

print("CONSOLIDATED NUMERICAL RESULTS: PASS")

# ------------------------------------------------------------
# C. VERIFY OPEN METHODOLOGICAL ISSUES
# ------------------------------------------------------------

issue_list = issues_data["issues"]

assert len(issue_list) == 4
assert {x["id"] for x in issue_list} == EXPECTED_ISSUES
assert all(x["resolved"] is False for x in issue_list)

assert a4["unresolved_methodological_issues"] == 4
assert a4["supervisor_review_required"] is True

open_issues = {
    "status": "OPEN_SUPERVISOR_REVIEW_REQUIRED",
    "inherited_from": rel(ISSUES),
    "issues": issue_list,
    "issue_count": 4,
    "all_unresolved": True,
    "report_final_approval": False
}

print("FOUR OPEN METHODOLOGICAL ISSUES: PRESERVED")

# ------------------------------------------------------------
# D. VERIFY AND RENUMBER TEN ORIGINAL FIGURES
# ------------------------------------------------------------

main_figures = figures[
    figures["selection"] == "MAIN"
].copy()

assert len(main_figures) == 10
assert set(main_figures["figure_id"]) == set(MAIN_ORDER)

by_id = main_figures.set_index("figure_id")

figure_rows = []
figure_hashes = {}

for index, fid in enumerate(MAIN_ORDER, start=1):
    item = by_id.loc[fid]

    image_path = (
        ROOT / str(item["original_repository_path"])
    ).resolve()

    assert image_path.is_relative_to(ROOT.resolve())
    assert image_path.is_file(), f"Missing PNG: {image_path}"

    with image_path.open("rb") as f:
        assert f.read(8) == bytes.fromhex(
            "89504e470d0a1a0a"
        )

    image_hash = sha256(image_path)

    assert image_hash == item["source_sha256"]
    assert image_hash == a3["original_figure_sha256"][
        rel(image_path)
    ]

    section = str(item["suggested_chapter_section"])
    number = f"Gambar 4.{index}"

    figure_rows.append({
        "figure_number": number,
        "figure_id": fid,
        "chapter_section": section,
        "caption": item["academic_caption"],
        "interpretation_note": item["interpretation_note"],
        "original_path": rel(image_path),
        "markdown_image_link": md_image_link(image_path),
        "sha256": image_hash
    })

    figure_hashes[rel(image_path)] = image_hash

figure_order = pd.DataFrame(figure_rows)

# Ensure actual document order is 4.3 -> 4.5 -> 4.6.
expected_sections = ["4.3"] * 3 + ["4.5"] * 6 + ["4.6"]
assert figure_order["chapter_section"].tolist() == (
    expected_sections
)

def figure_block(fid):
    row = figure_order.loc[
        figure_order["figure_id"] == fid
    ].iloc[0]

    return (
        f"### {row['figure_number']}\n\n"
        f"![{row['figure_number']}]"
        f"({row['markdown_image_link']})\n\n"
        f"**{row['figure_number']}. {row['caption']}**\n\n"
        f"*Catatan interpretasi:* "
        f"{row['interpretation_note']}\n\n"
        f"*Sumber:* `{row['original_path']}`\n"
    )

print("TEN ORIGINAL FIGURES HASHED AND RENUMBERED: PASS")

# ------------------------------------------------------------
# E. BUILD RESEARCH QUESTION EVIDENCE MATRIX
# ------------------------------------------------------------

evidence = questions.copy()

evidence["part5_evidence_reference"] = (
    evidence["research_question_id"].map({
        "RM1": "Section 4.2, 4.3",
        "RM2": "Section 4.3, 4.4",
        "RM3": "Section 4.5, 4.6"
    })
)

evidence["supervisor_review_required"] = True
evidence["final_approval"] = False

assert evidence["part5_evidence_reference"].notna().all()

# ------------------------------------------------------------
# F. BUILD TABLES FOR THE INTEGRATED REPORT
# ------------------------------------------------------------

data_view = dataset[[
    "case_name", "total_tweets",
    "count_NETRAL", "count_POSITIF", "count_NEGATIF"
]].rename(columns={
    "case_name": "Kasus",
    "total_tweets": "Tweet",
    "count_NETRAL": "Netral",
    "count_POSITIF": "Positif",
    "count_NEGATIF": "Negatif"
})

model_view = models[[
    "model", "accuracy",
    "macro_precision", "macro_recall",
    "macro_f1", "weighted_f1"
]].rename(columns={
    "model": "Model",
    "accuracy": "Accuracy",
    "macro_precision": "Macro-Precision",
    "macro_recall": "Macro-Recall",
    "macro_f1": "Macro-F1",
    "weighted_f1": "Weighted-F1"
})

error_view = errors[[
    "model", "sentiment_label", "support",
    "true_positive", "false_positive",
    "false_negative", "precision", "recall", "f1"
]].rename(columns={
    "model": "Model",
    "sentiment_label": "Sentimen",
    "support": "Support",
    "true_positive": "TP",
    "false_positive": "FP",
    "false_negative": "FN",
    "precision": "Precision",
    "recall": "Recall",
    "f1": "F1"
})

temp_view = temporal[[
    "case_name", "total_tweets",
    "calendar_days", "observed_days",
    "zero_tweet_days", "peak_date_wib",
    "peak_daily_tweets"
]].rename(columns={
    "case_name": "Kasus",
    "total_tweets": "Tweet",
    "calendar_days": "Hari kalender",
    "observed_days": "Hari observasi",
    "zero_tweet_days": "Hari kosong",
    "peak_date_wib": "Tanggal puncak (WIB)",
    "peak_daily_tweets": "Tweet puncak"
})

composition_view = temporal[[
    "case_name", "pct_NETRAL",
    "pct_POSITIF", "pct_NEGATIF"
]].rename(columns={
    "case_name": "Kasus",
    "pct_NETRAL": "Netral (%)",
    "pct_POSITIF": "Positif (%)",
    "pct_NEGATIF": "Negatif (%)"
})

def section_figures(ids):
    return "\n\n".join(figure_block(fid) for fid in ids)

# ------------------------------------------------------------
# G. INTEGRATED PROVISIONAL RESEARCH REPORT
# ------------------------------------------------------------

report = f"""# BAB IV — HASIL PENELITIAN DAN PEMBAHASAN

**Penelitian:** Sentimen Penetapan Bencana Nasional

**Status:** PROVISIONAL — Menunggu penyelarasan pembimbing.

**Tahap:** 06_Result Part5

**Dibuat (UTC):** {datetime.now(timezone.utc).isoformat()}

> Dokumen ini mengintegrasikan hasil penelitian yang
> sudah dikunci. Empat isu metodologis masih terbuka.
> Belum merupakan BAB IV final yang disetujui.

## 4.1 Gambaran Umum Dataset

Penelitian menggunakan **348 tweet unik** yang telah
memiliki label sentimen manusia, mencakup tiga kasus
bencana: Lombok 2018, Palu 2018, dan Sumatera 2025.

### Tabel 4.1 Distribusi Dataset

{md_table(data_view)}

Secara keseluruhan, terdapat **225 tweet NETRAL**
({225/348*100:.2f}%), **108 tweet POSITIF**
({108/348*100:.2f}%), dan **15 tweet NEGATIF**
({15/348*100:.2f}%).

Jumlah kelompok *leakage* unik pada dataset adalah 337.
Dominasi kelas NETRAL dan sedikitnya data NEGATIF
menjadi pertimbangan penting dalam evaluasi.

Hasil menggambarkan sampel yang terkumpul dan
tidak secara otomatis merepresentasikan seluruh
opini masyarakat.

## 4.2 Pengaturan Evaluasi Model

Penelitian mengimplementasikan model
**Long Short-Term Memory (LSTM)** untuk klasifikasi
sentimen tiga kelas.

Evaluasi menggunakan **Stratified Group K-Fold**
dengan tiga fold dan seed 123.
Prediksi *out-of-fold* digunakan untuk menghitung
metrik evaluasi pada 348 tweet.

TF-IDF + Logistic Regression digunakan sebagai
baseline pembanding tambahan.

Metrik utama yang digunakan adalah **OOF Macro-F1**,
agar ketiga kelas memperoleh bobot yang sama.

Tidak digunakan *independent holdout test set*.
Seluruh hasil evaluasi perlu ditafsirkan sesuai
dengan rancangan cross-validation tersebut.

**Catatan:** keberadaan baseline belum selaras
dengan batasan metode dalam proposal awal
dan memerlukan konfirmasi pembimbing.

## 4.3 Hasil Pemodelan Sentimen

### Tabel 4.2 Perbandingan Kinerja Model

{md_table(model_view)}

Model LSTM menghasilkan accuracy
**{l["accuracy"]*100:.2f}%**
dengan Macro-F1 **{l["macro_f1"]:.4f}**.

Baseline menghasilkan accuracy
**{b["accuracy"]*100:.2f}%**
dan Macro-F1 **{b["macro_f1"]:.4f}**.

Perbedaan Macro-F1 adalah
**{b["macro_f1"]-l["macro_f1"]:.4f} poin**
dengan baseline memiliki nilai lebih tinggi.

Namun, Macro-Recall LSTM
(**{l["macro_recall"]:.4f}**) lebih tinggi daripada
baseline (**{b["macro_recall"]:.4f}**).
Hasil ini menunjukkan perbedaan karakteristik
evaluasi kedua model, bukan keunggulan LSTM
di seluruh metrik.

{section_figures(["MODEL_G05", "MODEL_G04", "MODEL_G06"])}

## 4.4 Analisis Kesalahan Klasifikasi

### Tabel 4.3 Evaluasi per Kelas

{md_table(error_view)}

Kelas NEGATIF hanya terdiri atas 15 tweet aktual.
Baseline mengenali
**{int(neg.loc["baseline","true_positive"])}**
tweet NEGATIF dan melewatkan
**{int(neg.loc["baseline","false_negative"])}**.

LSTM mengenali
**{int(neg.loc["lstm","true_positive"])}**
tweet NEGATIF, tetapi menghasilkan
**{int(neg.loc["lstm","false_positive"])}**
*false positive* NEGATIF.

Recall NEGATIF LSTM adalah
**{neg.loc["lstm","recall"]*100:.2f}%**,
sedangkan precision hanya
**{neg.loc["lstm","precision"]*100:.2f}%**.

Artinya, LSTM lebih banyak menangkap contoh
NEGATIF yang sebenarnya, tetapi juga lebih sering
memberikan label NEGATIF kepada tweet yang bukan
anggota kelas tersebut.

Perbedaan tersebut perlu dibahas sebagai
*trade-off* precision dan recall.

Berdasarkan perbandingan prediksi berpasangan,
kedua model benar pada 165 tweet,
hanya baseline benar pada 80 tweet,
hanya LSTM benar pada 39 tweet,
dan keduanya salah pada 64 tweet.

Angka ini tidak digunakan untuk menyatakan
signifikansi statistik tanpa uji tambahan.

## 4.5 Distribusi Sentimen dan Analisis Temporal

Analisis temporal menggunakan **label manusia**
dan timestamp tweet yang dikonversi ke WIB.
Analisis ini **tidak menggunakan hasil prediksi LSTM**
sebagai sumber label timeline.

### Tabel 4.4 Cakupan Observasi

{md_table(temp_view)}

Total terdapat **20 hari kalender kasus**,
yaitu 15 hari dengan observasi dan lima hari
tanpa tweet dalam sampel.

Puncak harian Lombok terjadi pada
2018-08-10 dengan 24 tweet;
Palu pada 2018-10-01 dengan 36 tweet;
dan Sumatera pada 2025-12-03 dengan 54 tweet.

Periode observasi masing-masing kasus berbeda.
Oleh karena itu, volume antarkasus tidak dapat
langsung diartikan sebagai perbedaan tingkat
perhatian publik.

### 4.5.1 Jumlah Tweet Harian

{section_figures(["TEMP_T04_F01", "TEMP_T04_F02",
                  "TEMP_T04_F03"])}

### 4.5.2 Proporsi Sentimen Harian

{section_figures(["TEMP_T04_F04", "TEMP_T04_F05",
                  "TEMP_T04_F06"])}

Persentase pada hari tanpa observasi
bersifat tidak terdefinisi, bukan nol.
Pembacaan proporsi harian perlu mempertimbangkan
jumlah tweet yang tersedia pada tanggal tersebut.

## 4.6 Perbandingan Sentimen Antarkasus

### Tabel 4.5 Komposisi Sentimen per Kasus

{md_table(composition_view)}

NETRAL merupakan kategori terbesar
pada ketiga kasus.

Persentase NETRAL masing-masing kasus adalah:
Lombok **{float(temporal.loc[temporal["case"]=="lombok_2018","pct_NETRAL"].iloc[0]):.2f}%**,
Palu **{float(temporal.loc[temporal["case"]=="palu_2018","pct_NETRAL"].iloc[0]):.2f}%**,
dan Sumatera **{float(temporal.loc[temporal["case"]=="sumatra_2025","pct_NETRAL"].iloc[0]):.2f}%**.

Perbedaan komposisi ini merupakan temuan
deskriptif atas tweet yang dianotasi,
bukan bukti pengaruh kausal keputusan kebijakan.

{section_figures(["TEMP_T04_F08"])}

## 4.7 Kesesuaian dengan Rumusan Masalah

### RM1 — Penerapan LSTM

RM1 dijawab melalui implementasi LSTM dan evaluasi
OOF pada tiga kategori sentimen.
Kinerja yang diperoleh mencakup accuracy,
Macro-F1, dan metrik per kelas.

### RM2 — Hasil Klasifikasi Tiga Kelas

RM2 didukung oleh hasil klasifikasi LSTM,
confusion matrix, serta pengukuran precision,
recall, dan F1 pada NETRAL, POSITIF, dan NEGATIF.

Keterbatasan paling menonjol terjadi pada
kelas NEGATIF yang memiliki 15 tweet aktual.

### RM3 — Dinamika Temporal Sentimen

RM3 dijawab secara **deskriptif** melalui distribusi
label manusia per tanggal pada tiga kasus.

Hasil ini belum dapat disebut sebagai
dinamika sentimen yang *diprediksi oleh LSTM*.

Perlu konfirmasi apakah analisis berbasis label
manusia sudah sesuai ruang lingkup yang disetujui
atau membutuhkan evaluasi temporal model tambahan.

## 4.8 Keterbatasan dan Isu Metodologis

Analisis memiliki beberapa keterbatasan:
ukuran sampel 348 tweet, distribusi label timpang,
tidak adanya independent holdout test set,
serta perbedaan jendela observasi antarkasus.

Empat isu metodologis dari Part4 tetap terbuka:

"""

for issue in issue_list:
    report += (
        f"\n### {issue['id']} — {issue['topic']}\n\n"
        f"**Prioritas:** {issue['severity']}\n\n"
        f"**Masalah:** {issue['implementation']}\n\n"
        f"**Tindakan:** {issue['required_action']}\n\n"
        "**Status:** Belum terselesaikan.\n"
    )

report += f"""

## 4.9 Ringkasan Hasil Penelitian

Berdasarkan evaluasi OOF, baseline memiliki
Macro-F1 lebih tinggi daripada LSTM, sedangkan
LSTM memiliki Macro-Recall lebih tinggi.

Analisis kesalahan menunjukkan tantangan
identifikasi kelas NEGATIF karena data aktual
yang terbatas serta perbedaan precision dan recall.

Pada analisis temporal berbasis label manusia,
NETRAL mendominasi sampel pada tiga kasus,
dengan variasi distribusi dan cakupan waktu.

Seluruh kesimpulan berlaku pada dataset penelitian.
Tidak dilakukan inferensi populasi maupun
penarikan kesimpulan kausal atas kebijakan.

**Status dokumen: PROVISIONAL.**

**Empat isu metodologis terbuka: {len(issue_list)}.**

**Persetujuan pembimbing: BELUM TERKONFIRMASI.**

## Catatan Provenance

- Dataset dan metrik telah diaudit pada Part1–Part2.
- Gambar merupakan PNG asli yang dikunci pada Part3.
- Rumusan masalah dipetakan dalam Part4.
- Semua nomor gambar pada Part5 mengikuti urutan
  subbab dan masih dapat disesuaikan dengan
  pedoman penulisan akhir.
- Tidak ada model yang dilatih ulang dalam Part5.
"""

# ------------------------------------------------------------
# H. PREPARE OUTPUT BYTES + AUDIT
# ------------------------------------------------------------

out_bytes = {
    REPORT_OUT: report.encode("utf-8"),
    ORDER_OUT: figure_order.to_csv(
        index=False
    ).encode("utf-8-sig"),
    EVIDENCE_OUT: evidence.to_csv(
        index=False
    ).encode("utf-8-sig"),
    ISSUES_OUT: json_bytes(open_issues)
}

audit5 = {
    "stage": "06_Result",
    "part": "5 Provisional Integrated Research Report",
    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "source_commit":
        "a705a32160d25733bf7c5083dfe0b8525590ab41",
    "source_sha256": source_hashes,
    "original_figure_sha256": figure_hashes,
    "dataset_rows": 348,
    "models": 2,
    "cases": 3,
    "research_questions": 3,
    "main_figures_embedded": 10,
    "figures_recreated": False,
    "figure_order_corrected": True,
    "open_methodological_issues": 4,
    "supervisor_review_required": True,
    "final_academic_approval": False,
    "model_training_performed": False,
    "source_modified": False,
    "output_sha256": {
        p.name: hashlib.sha256(blob).hexdigest()
        for p, blob in out_bytes.items()
    },
    "status": "RESULT_PART5_PROVISIONAL_REPORT_PASS"
}

out_bytes[AUDIT_OUT] = json_bytes(audit5)

# ------------------------------------------------------------
# I. NO-OVERWRITE EXPORT
# ------------------------------------------------------------

OUT.mkdir(parents=True, exist_ok=True)

for p in OUTPUTS:
    if p.exists():
        raise FileExistsError(f"No overwrite: {p}")

created = []

try:
    assert {
        rel(p): sha256(p) for p in SOURCES
    } == source_hashes

    assert {
        path: sha256(ROOT / path)
        for path in figure_hashes
    } == figure_hashes

    # Verify all expected figures referenced exactly once.
    for row in figure_order.itertuples(index=False):
        assert report.count(
            f"![{row.figure_number}]"
        ) == 1

    for p in OUTPUTS:
        with p.open("xb") as f:
            f.write(out_bytes[p])
        created.append(p)

    for p, blob in out_bytes.items():
        assert sha256(p) == hashlib.sha256(
            blob
        ).hexdigest()

    assert len(pd.read_csv(
        ORDER_OUT, encoding="utf-8-sig"
    )) == 10

    assert len(pd.read_csv(
        EVIDENCE_OUT, encoding="utf-8-sig"
    )) == 3

    assert read_json(AUDIT_OUT)["status"] == (
        "RESULT_PART5_PROVISIONAL_REPORT_PASS"
    )

    assert {
        rel(p): sha256(p) for p in SOURCES
    } == source_hashes

except Exception:
    # Delete only files created during the failed run.
    for p in created:
        if p.exists():
            p.unlink()
    raise

# ------------------------------------------------------------
# J. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("06_RESULT PART5 — FINAL AUDIT")
print("=" * 72)

print("Dataset tweets       : 348")
print("Models referenced    : 2")
print("Disaster cases       : 3")
print("Research questions   : 3")
print("Main figures         : 10")
print("Open methodology     : 4")
print("Academic status      : PROVISIONAL")
print("Source modification  : NO")
print("Model training       : NO")

print("\nFIGURE ORDER")
display(figure_order[[
    "figure_number", "figure_id", "chapter_section"
]])

print("\nOUTPUT FILES")
for p in OUTPUTS:
    print("-", p)

print("\nFINAL STATUS:", audit5["status"])
print("=" * 72)


06_RESULT PART5 — INTEGRATED REPORT PREFLIGHT
PART2 / PART3 / PART4 AUDIT CHAIN: PASS
CONSOLIDATED NUMERICAL RESULTS: PASS
FOUR OPEN METHODOLOGICAL ISSUES: PRESERVED
TEN ORIGINAL FIGURES HASHED AND RENUMBERED: PASS

06_RESULT PART5 — FINAL AUDIT
Dataset tweets       : 348
Models referenced    : 2
Disaster cases       : 3
Research questions   : 3
Main figures         : 10
Open methodology     : 4
Academic status      : PROVISIONAL
Source modification  : NO
Model training       : NO

FIGURE ORDER


,figure_number,figure_id,chapter_section
0,Gambar 4.1,MODEL_G05,4.3
1,Gambar 4.2,MODEL_G04,4.3
2,Gambar 4.3,MODEL_G06,4.3
3,Gambar 4.4,TEMP_T04_F01,4.5
4,Gambar 4.5,TEMP_T04_F02,4.5
5,Gambar 4.6,TEMP_T04_F03,4.5
6,Gambar 4.7,TEMP_T04_F04,4.5
7,Gambar 4.8,TEMP_T04_F05,4.5
8,Gambar 4.9,TEMP_T04_F06,4.5
9,Gambar 4.10,TEMP_T04_F08,4.6



OUTPUT FILES
- C:\Penelitian Sentimen Bencana Nasional\06_Result\outputs\integrated_report\provisional_integrated_research_report.md
- C:\Penelitian Sentimen Bencana Nasional\06_Result\outputs\integrated_report\final_main_figure_order_10.csv
- C:\Penelitian Sentimen Bencana Nasional\06_Result\outputs\integrated_report\research_question_evidence_matrix.csv
- C:\Penelitian Sentimen Bencana Nasional\06_Result\outputs\integrated_report\open_methodological_issues.json
- C:\Penelitian Sentimen Bencana Nasional\06_Result\outputs\integrated_report\part5_integrated_report_audit.json

FINAL STATUS: RESULT_PART5_PROVISIONAL_REPORT_PASS


In [7]:
# 06_RESULT — PART6 / CELL 06
# FINAL TECHNICAL AUDIT & PDF EXPORT
#
# Locked Part1-5 | ReportLab | 10 original PNGs
# 4 open issues retained | No training | No overwrite
# ==============================================================

from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import unquote
from html import escape
import hashlib
import json
import os
import re
import uuid

import pandas as pd
import markdown
from bs4 import BeautifulSoup, NavigableString, Tag

from PIL import Image as PILImage
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer,
    Image, LongTable, TableStyle,
    KeepTogether, PageBreak
)
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_JUSTIFY, TA_LEFT
from reportlab.lib.units import mm
from reportlab.pdfbase.pdfmetrics import stringWidth

# --------------------------------------------------------------
# A. PATH CONFIGURATION
# --------------------------------------------------------------

ROOT = Path(r"C:\Penelitian Sentimen Bencana Nasional")
BASE = ROOT / "06_Result" / "outputs"

P1 = BASE / "cross_phase_integrity"
P2 = BASE / "consolidated_results"
P3 = BASE / "figure_selection"
P4 = BASE / "discussion_alignment"
P5 = BASE / "integrated_report"

OUT = BASE / "final_technical_report"

AUDIT1 = P1 / "part1_cross_phase_integrity_audit.json"
AUDIT2 = P2 / "part2_consolidated_results_audit.json"
AUDIT3 = P3 / "part3_figure_selection_audit.json"
AUDIT4 = P4 / "part4_discussion_alignment_audit.json"
AUDIT5 = P5 / "part5_integrated_report_audit.json"

REPORT_MD = P5 / "provisional_integrated_research_report.md"
FIGURE_ORDER = P5 / "final_main_figure_order_10.csv"
EVIDENCE = P5 / "research_question_evidence_matrix.csv"
ISSUES = P5 / "open_methodological_issues.json"

MODEL = P2 / "consolidated_model_evaluation.csv"
DATA = P2 / "dataset_characteristics.csv"
ERRORS = P2 / "model_error_analysis_by_class.csv"
TEMP = P2 / "consolidated_temporal_findings.csv"

PDF_OUT = OUT / "provisional_integrated_research_report.pdf"
FIG_MANIFEST_OUT = OUT / "final_pdf_figure_manifest.csv"
AUDIT_OUT = OUT / "part6_final_technical_audit.json"
HANDOFF_OUT = OUT / "part6_technical_handoff.md"

OUTPUTS = [
    PDF_OUT,
    FIG_MANIFEST_OUT,
    AUDIT_OUT,
    HANDOFF_OUT
]

SOURCES = [
    AUDIT1, AUDIT2, AUDIT3, AUDIT4, AUDIT5,
    REPORT_MD, FIGURE_ORDER, EVIDENCE, ISSUES,
    MODEL, DATA, ERRORS, TEMP
]

EXPECTED_ISSUES = {
    "METHOD_01",
    "METHOD_02",
    "METHOD_03",
    "METHOD_04"
}

# --------------------------------------------------------------
# B. UTILITY FUNCTIONS
# --------------------------------------------------------------

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def rel(path):
    return path.resolve().relative_to(ROOT.resolve()).as_posix()

def json_bytes(data):
    return json.dumps(
        data,
        ensure_ascii=False,
        indent=2,
        allow_nan=False
    ).encode("utf-8")

def verify_output_of(path, audit):
    expected = audit["output_sha256"].get(path.name)
    assert expected, f"Missing source hash entry: {path.name}"
    assert sha256(path) == expected, (
        f"SHA256 mismatch: {rel(path)}"
    )

def safe_text(text):
    return escape(str(text), quote=False)

def inline_markup(node):
    """Convert basic Markdown HTML to ReportLab-safe markup."""
    if isinstance(node, NavigableString):
        return safe_text(str(node))

    if not isinstance(node, Tag):
        return ""

    children = "".join(inline_markup(x) for x in node.children)

    if node.name in ("strong", "b"):
        return f"<b>{children}</b>"
    if node.name in ("em", "i"):
        return f"<i>{children}</i>"
    if node.name == "code":
        return f'<font color="#334155">{children}</font>'
    if node.name == "br":
        return " "

    # Links are rendered as plain text; sources remain in Markdown.
    return children

print("=" * 76)
print("06_RESULT PART6 — FINAL TECHNICAL PREFLIGHT")
print("=" * 76)

# --------------------------------------------------------------
# C. VERIFY LOCKED SOURCE AND AUDIT CHAIN
# --------------------------------------------------------------

for path in SOURCES:
    assert path.is_file(), f"Source missing: {path}"

for path in OUTPUTS:
    if path.exists():
        raise FileExistsError(
            f"Part6 output already exists; refusing overwrite: {path}"
        )

a1, a2, a3, a4, a5 = [
    read_json(p)
    for p in [AUDIT1, AUDIT2, AUDIT3, AUDIT4, AUDIT5]
]

expected_statuses = [
    "RESULT_PART1_CROSS_PHASE_INTEGRITY_PASS",
    "RESULT_PART2_CONSOLIDATED_RESULTS_PASS",
    "RESULT_PART3_FIGURE_SELECTION_PASS",
    "RESULT_PART4_DRAFT_ALIGNMENT_PASS",
    "RESULT_PART5_PROVISIONAL_REPORT_PASS"
]

for audit, expected_status in zip(
    [a1, a2, a3, a4, a5],
    expected_statuses
):
    assert audit["status"] == expected_status

# Verify audit-to-audit continuity.
assert sha256(AUDIT2) == a3["source_sha256"][rel(AUDIT2)]
assert sha256(AUDIT3) == a4["source_sha256"][rel(AUDIT3)]
assert sha256(AUDIT4) == a5["source_sha256"][rel(AUDIT4)]

# Verify all Part5 output artifacts.
for path in [REPORT_MD, FIGURE_ORDER, EVIDENCE, ISSUES]:
    verify_output_of(path, a5)

# Verify consolidated numerical sources.
for path in [MODEL, DATA, ERRORS, TEMP]:
    verify_output_of(path, a2)
    assert sha256(path) == a5["source_sha256"][rel(path)]

source_before = {
    rel(p): sha256(p)
    for p in SOURCES
}

print("PART1-PART5 AUDIT STATUSES: PASS")
print("CROSS-STAGE SHA256 CONTINUITY: PASS")

# --------------------------------------------------------------
# D. VERIFY RESEARCH RESULTS AND OPEN ISSUES
# --------------------------------------------------------------

models = pd.read_csv(MODEL, encoding="utf-8-sig")
dataset = pd.read_csv(DATA, encoding="utf-8-sig")
errors = pd.read_csv(ERRORS, encoding="utf-8-sig")
temporal = pd.read_csv(TEMP, encoding="utf-8-sig")
figures = pd.read_csv(FIGURE_ORDER, encoding="utf-8-sig")
evidence = pd.read_csv(EVIDENCE, encoding="utf-8-sig")

issues_data = read_json(ISSUES)
issue_list = issues_data["issues"]

assert len(models) == 2
assert len(dataset) == 3
assert len(errors) == 6
assert len(temporal) == 3
assert len(figures) == 10
assert len(evidence) == 3
assert len(issue_list) == 4

assert int(dataset["total_tweets"].sum()) == 348
assert int(temporal["total_tweets"].sum()) == 348
assert int(temporal["calendar_days"].sum()) == 20
assert int(temporal["observed_days"].sum()) == 15
assert int(temporal["zero_tweet_days"].sum()) == 5

for label, count in [
    ("NETRAL", 225),
    ("POSITIF", 108),
    ("NEGATIF", 15)
]:
    assert int(dataset[f"count_{label}"].sum()) == count
    assert int(temporal[f"count_{label}"].sum()) == count

m = models.set_index("model")
assert set(m.index) == {"baseline", "lstm"}

assert abs(float(m.loc["baseline", "macro_f1"])
           - 0.5235550389755573) < 1e-12
assert abs(float(m.loc["lstm", "macro_f1"])
           - 0.47512961484120647) < 1e-12

assert set(evidence["research_question_id"]) == {
    "RM1", "RM2", "RM3"
}
assert evidence["supervisor_review_required"].all()
assert not evidence["final_approval"].any()

assert {x["id"] for x in issue_list} == EXPECTED_ISSUES
assert all(x["resolved"] is False for x in issue_list)
assert issues_data["report_final_approval"] is False
assert a5["final_academic_approval"] is False

print("348-TWEET NUMERICAL RESULTS: PASS")
print("RM1-RM3 EVIDENCE: PASS")
print("4 UNRESOLVED METHODOLOGICAL ISSUES: PRESERVED")

# --------------------------------------------------------------
# E. VERIFY TEN ORIGINAL FIGURES
# --------------------------------------------------------------

expected_sections = ["4.3"] * 3 + ["4.5"] * 6 + ["4.6"]

assert figures["chapter_section"].astype(str).tolist() == (
    expected_sections
)
assert figures["figure_number"].tolist() == [
    f"Gambar 4.{i}" for i in range(1, 11)
]
assert figures["figure_id"].is_unique

image_lookup = {}
figure_hashes = {}
manifest_rows = []

for row in figures.itertuples(index=False):
    img = (ROOT / row.original_path).resolve()

    assert img.is_relative_to(ROOT.resolve())
    assert img.is_file(), f"Image not found: {img}"

    expected_hash = row.sha256
    actual_hash = sha256(img)

    assert actual_hash == expected_hash
    assert actual_hash == a5["original_figure_sha256"][
        rel(img)
    ]

    with PILImage.open(img) as im:
        assert im.format == "PNG"
        width, height = im.size
        im.verify()

    image_lookup[rel(img)] = img
    figure_hashes[rel(img)] = actual_hash

    manifest_rows.append({
        "figure_number": row.figure_number,
        "figure_id": row.figure_id,
        "chapter_section": row.chapter_section,
        "original_path": rel(img),
        "sha256": actual_hash,
        "width_px": width,
        "height_px": height,
        "file_size_bytes": img.stat().st_size
    })

figure_manifest = pd.DataFrame(manifest_rows)

assert len(figure_manifest) == 10
assert len(figure_hashes) == 10

print("TEN ORIGINAL PNG FILES AND SHA256: PASS")

# --------------------------------------------------------------
# F. VALIDATE MARKDOWN REPORT STRUCTURE
# --------------------------------------------------------------

md_text = REPORT_MD.read_text(encoding="utf-8")

assert "PROVISIONAL" in md_text
assert "Persetujuan pembimbing" in md_text
assert "RM1" in md_text
assert "RM2" in md_text
assert "RM3" in md_text

for issue_id in sorted(EXPECTED_ISSUES):
    assert issue_id in md_text

for i in range(1, 11):
    assert md_text.count(f"![Gambar 4.{i}]") == 1

for section in range(1, 10):
    assert f"## 4.{section} " in md_text

html = markdown.markdown(
    md_text,
    extensions=["tables", "fenced_code"]
)
soup = BeautifulSoup(html, "html.parser")

images_in_md = soup.find_all("img")
assert len(images_in_md) == 10

resolved_images = []

for tag in images_in_md:
    src = unquote(tag.get("src", ""))
    image_path = (REPORT_MD.parent / src).resolve()

    assert image_path.is_relative_to(ROOT.resolve())
    assert rel(image_path) in image_lookup

    resolved_images.append(rel(image_path))

assert set(resolved_images) == set(image_lookup)
assert len(set(resolved_images)) == 10

print("MARKDOWN, 9 SECTIONS, 10 IMAGE REFERENCES: PASS")

# --------------------------------------------------------------
# G. REPORTLAB STYLES AND PAGE SETTINGS
# --------------------------------------------------------------

PAGE_W, PAGE_H = A4
LEFT = RIGHT = 17 * mm
TOP = 19 * mm
BOTTOM = 18 * mm
CONTENT_W = PAGE_W - LEFT - RIGHT

styles = getSampleStyleSheet()

styles.add(ParagraphStyle(
    name="ResearchTitle",
    fontName="Helvetica-Bold",
    fontSize=16,
    leading=21,
    alignment=TA_CENTER,
    spaceAfter=15
))

styles.add(ParagraphStyle(
    name="ResearchH2",
    fontName="Helvetica-Bold",
    fontSize=12,
    leading=16,
    spaceBefore=12,
    spaceAfter=8,
    keepWithNext=True
))

styles.add(ParagraphStyle(
    name="ResearchH3",
    fontName="Helvetica-Bold",
    fontSize=10,
    leading=14,
    spaceBefore=9,
    spaceAfter=6,
    keepWithNext=True
))

styles.add(ParagraphStyle(
    name="ResearchBody",
    fontName="Helvetica",
    fontSize=9.2,
    leading=14,
    alignment=TA_JUSTIFY,
    spaceAfter=7
))

styles.add(ParagraphStyle(
    name="ResearchCaption",
    fontName="Helvetica",
    fontSize=8.2,
    leading=11.5,
    spaceAfter=9
))

styles.add(ParagraphStyle(
    name="ResearchTable",
    fontName="Helvetica",
    fontSize=6.7,
    leading=9,
    alignment=TA_LEFT
))

styles.add(ParagraphStyle(
    name="ResearchQuote",
    parent=styles["ResearchBody"],
    leftIndent=10 * mm,
    textColor=colors.HexColor("#475569")
))

def pdf_footer(canvas, doc):
    canvas.saveState()
    canvas.setFont("Helvetica", 7.5)
    canvas.setFillColor(colors.grey)

    canvas.drawString(
        LEFT, 11 * mm,
        "BAB IV | PROVISIONAL"
    )
    canvas.drawRightString(
        PAGE_W - RIGHT, 11 * mm,
        f"Halaman {doc.page}"
    )
    canvas.restoreState()

# --------------------------------------------------------------
# H. MARKDOWN TO PDF FLOWABLES
# --------------------------------------------------------------

story = []
rendered_image_count = 0
rendered_table_count = 0

def paragraph_for(tag, style):
    markup = inline_markup(tag)
    return Paragraph(markup or " ", style)

def add_image(img_tag):
    global rendered_image_count

    src = unquote(img_tag.get("src", ""))
    path = (REPORT_MD.parent / src).resolve()

    with PILImage.open(path) as im:
        w, h = im.size

    # Keep figure inside the printable area.
    max_w = CONTENT_W
    max_h = 145 * mm

    ratio = min(
        max_w / w,
        max_h / h
    )

    pdf_img = Image(
        str(path),
        width=w * ratio,
        height=h * ratio
    )

    pdf_img.hAlign = "CENTER"

    story.append(Spacer(1, 2 * mm))
    story.append(pdf_img)
    story.append(Spacer(1, 2 * mm))

    rendered_image_count += 1

def add_table(table_tag):
    global rendered_table_count

    rows = []

    for tr in table_tag.find_all("tr"):
        cells = tr.find_all(
            ["th", "td"], recursive=False
        )
        if not cells:
            continue

        rows.append([
            paragraph_for(c, styles["ResearchTable"])
            for c in cells
        ])

    if not rows:
        return

    num_cols = max(len(x) for x in rows)

    for row in rows:
        row.extend([""] * (num_cols - len(row)))

    col_widths = [CONTENT_W / num_cols] * num_cols

    pdf_table = LongTable(
        rows,
        colWidths=col_widths,
        repeatRows=1,
        hAlign="CENTER"
    )

    pdf_table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0),
         colors.HexColor("#E8EDF5")),
        ("GRID", (0, 0), (-1, -1),
         0.3, colors.HexColor("#A0A0A0")),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("LEFTPADDING", (0, 0), (-1, -1), 3),
        ("RIGHTPADDING", (0, 0), (-1, -1), 3),
        ("TOPPADDING", (0, 0), (-1, -1), 4),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 4),
    ]))

    story.append(Spacer(1, 3 * mm))
    story.append(pdf_table)
    story.append(Spacer(1, 4 * mm))

    rendered_table_count += 1

for tag in soup.children:
    if not isinstance(tag, Tag):
        continue

    if tag.name == "h1":
        story.append(
            paragraph_for(tag, styles["ResearchTitle"])
        )

    elif tag.name == "h2":
        story.append(
            paragraph_for(tag, styles["ResearchH2"])
        )

    elif tag.name in ("h3", "h4"):
        story.append(
            paragraph_for(tag, styles["ResearchH3"])
        )

    elif tag.name == "p":
        images = tag.find_all("img")
        if images:
            for image_tag in images:
                add_image(image_tag)
        else:
            content = tag.get_text(" ", strip=True)

            style = (
                styles["ResearchCaption"]
                if re.match(r"^Gambar 4\.\d+\.", content)
                else styles["ResearchBody"]
            )

            story.append(paragraph_for(tag, style))

    elif tag.name == "table":
        add_table(tag)

    elif tag.name in ("ul", "ol"):
        for i, li in enumerate(
            tag.find_all("li", recursive=False), 1
        ):
            prefix = f"{i}. " if tag.name == "ol" else "- "
            text = prefix + li.get_text(" ", strip=True)

            story.append(
                Paragraph(
                    safe_text(text),
                    styles["ResearchBody"]
                )
            )

    elif tag.name == "blockquote":
        story.append(
            paragraph_for(tag, styles["ResearchQuote"])
        )

assert rendered_image_count == 10
assert rendered_table_count == 5

print("PDF FLOWABLE CONTENT: PASS")
print("Figures:", rendered_image_count)
print("Tables :", rendered_table_count)

# --------------------------------------------------------------
# I. GENERATE PDF INTO TEMPORARY FILE
# --------------------------------------------------------------

OUT.mkdir(parents=True, exist_ok=True)

for path in OUTPUTS:
    if path.exists():
        raise FileExistsError(
            f"Refusing overwrite: {path}"
        )

token = uuid.uuid4().hex[:12]
TEMP_PDF = OUT / f".result_part6_{token}.pdf"

created = []

try:
    doc = SimpleDocTemplate(
        str(TEMP_PDF),
        pagesize=A4,
        leftMargin=LEFT,
        rightMargin=RIGHT,
        topMargin=TOP,
        bottomMargin=BOTTOM,
        title="BAB IV - Hasil Penelitian dan Pembahasan",
        author="Penelitian Sentimen Penetapan Bencana Nasional",
        subject="PROVISIONAL Integrated Research Results"
    )

    doc.build(
        story,
        onFirstPage=pdf_footer,
        onLaterPages=pdf_footer
    )

    assert TEMP_PDF.is_file()
    assert TEMP_PDF.stat().st_size > 10000

    with TEMP_PDF.open("rb") as f:
        assert f.read(5) == b"%PDF-"

    # Optional structural page validation, if installed.
    pdf_pages = None
    try:
        from pypdf import PdfReader
        reader = PdfReader(str(TEMP_PDF))
        pdf_pages = len(reader.pages)
        assert pdf_pages >= 2
    except ImportError:
        print(
            "INFO: pypdf not installed; "
            "PDF page count requires manual review."
        )

    # Verify source integrity before publishing.
    assert {
        rel(p): sha256(p)
        for p in SOURCES
    } == source_before

    assert {
        path: sha256(ROOT / path)
        for path in figure_hashes
    } == figure_hashes

    # ------------------------------------------------------
    # J. FINAL HANDOFF AND AUDIT
    # ------------------------------------------------------

    handoff = f"""# 06_RESULT PART6 — TECHNICAL HANDOFF

**Technical status:** PASS after successful export

**Academic status:** PROVISIONAL

## Research Scope

- 348 labeled tweets
- Three disaster cases
- Two evaluated models
- Three research questions
- Ten original figures in the PDF
- Four unresolved methodological issues

## Verified Source Artifacts

- Part1: Cross-Phase Integrity
- Part2: Consolidated Research Results
- Part3: Figure Selection
- Part4: Research Question Alignment
- Part5: Provisional Integrated Report

## Output

- PDF: `provisional_integrated_research_report.pdf`
- Figures: `final_pdf_figure_manifest.csv`
- Audit: `part6_final_technical_audit.json`

## Important Academic Restrictions

The report is not yet approved by the supervisor.

The following methodological issues remain open:

- METHOD_01: Baseline outside original proposal boundary
- METHOD_02: Human-label temporal analysis vs LSTM predictions
- METHOD_03: Policy terminology and causal interpretation
- METHOD_04: Dataset and evaluation limitations

## Final Human Quality Check

1. Open the exported PDF.
2. Verify all ten figures and five tables.
3. Check legends, page breaks, and captions.
4. Confirm the PROVISIONAL label is visible.
5. Confirm all four methodological issues remain open.
6. Obtain supervisor review before final academic submission.

**PDF page count:** {pdf_pages if pdf_pages else "Not checked automatically"}

No model training or source modification was performed.
"""

    manifest_bytes = figure_manifest.to_csv(
        index=False
    ).encode("utf-8-sig")

    handoff_bytes = handoff.encode("utf-8")
    pdf_bytes = TEMP_PDF.read_bytes()

    audit6 = {
        "stage": "06_Result",
        "part": "6 Final Technical Audit and PDF Export",
        "generated_at_utc":
            datetime.now(timezone.utc).isoformat(),
        "source_commit":
            "8f45939d8f7f7a74f24bc7158d3934af785da9cc",
        "source_audit_statuses": expected_statuses,
        "source_sha256": source_before,
        "original_figure_sha256": figure_hashes,
        "dataset_rows": 348,
        "research_questions": 3,
        "original_figures_in_pdf": 10,
        "tables_in_pdf": rendered_table_count,
        "pdf_page_count": pdf_pages,
        "pdf_signature_valid": True,
        "pdf_visual_review_complete": False,
        "methodological_issues_open": 4,
        "academic_status": "PROVISIONAL",
        "supervisor_approval_confirmed": False,
        "source_modified": False,
        "model_training_performed": False,
        "output_sha256": {
            PDF_OUT.name: hashlib.sha256(
                pdf_bytes
            ).hexdigest(),
            FIG_MANIFEST_OUT.name: hashlib.sha256(
                manifest_bytes
            ).hexdigest(),
            HANDOFF_OUT.name: hashlib.sha256(
                handoff_bytes
            ).hexdigest()
        },
        "status": "RESULT_PART6_TECHNICAL_AUDIT_PASS"
    }

    output_bytes = {
        PDF_OUT: pdf_bytes,
        FIG_MANIFEST_OUT: manifest_bytes,
        HANDOFF_OUT: handoff_bytes,
        AUDIT_OUT: json_bytes(audit6)
    }

    # Exclusive writes: no overwriting previous artifacts.
    for destination in OUTPUTS:
        with destination.open("xb") as f:
            f.write(output_bytes[destination])
        created.append(destination)

    # Check integrity of all exported files.
    for destination, data in output_bytes.items():
        assert sha256(destination) == (
            hashlib.sha256(data).hexdigest()
        )

    assert read_json(AUDIT_OUT)["status"] == (
        "RESULT_PART6_TECHNICAL_AUDIT_PASS"
    )

    assert {
        rel(p): sha256(p)
        for p in SOURCES
    } == source_before

    assert {
        path: sha256(ROOT / path)
        for path in figure_hashes
    } == figure_hashes

except Exception:
    # Remove only files created by this failed execution.
    for path in created:
        if path.exists():
            path.unlink()
    raise

finally:
    if TEMP_PDF.exists():
        TEMP_PDF.unlink()

# --------------------------------------------------------------
# K. FINAL OUTPUT
# --------------------------------------------------------------

print("\n" + "=" * 76)
print("06_RESULT PART6 — FINAL TECHNICAL AUDIT")
print("=" * 76)

print("Research tweets     : 348")
print("Research questions  : 3")
print("Models              : 2")
print("Cases               : 3")
print("Original figures    : 10")
print("PDF tables          :", rendered_table_count)
print("PDF pages           :", pdf_pages or "manual check")
print("Open methodology    : 4")
print("Academic approval   : NOT CONFIRMED")
print("Source modified     : NO")
print("Training performed  : NO")

print("\nOUTPUT ARTIFACTS:")
for path in OUTPUTS:
    print("-", path)

print("\nPDF SHA256:", sha256(PDF_OUT))
print("\nFINAL STATUS:", audit6["status"])
print("=" * 76)


06_RESULT PART6 — FINAL TECHNICAL PREFLIGHT
PART1-PART5 AUDIT STATUSES: PASS
CROSS-STAGE SHA256 CONTINUITY: PASS
348-TWEET NUMERICAL RESULTS: PASS
RM1-RM3 EVIDENCE: PASS
4 UNRESOLVED METHODOLOGICAL ISSUES: PRESERVED
TEN ORIGINAL PNG FILES AND SHA256: PASS
MARKDOWN, 9 SECTIONS, 10 IMAGE REFERENCES: PASS
PDF FLOWABLE CONTENT: PASS
Figures: 10
Tables : 5
INFO: pypdf not installed; PDF page count requires manual review.

06_RESULT PART6 — FINAL TECHNICAL AUDIT
Research tweets     : 348
Research questions  : 3
Models              : 2
Cases               : 3
Original figures    : 10
PDF tables          : 5
PDF pages           : manual check
Open methodology    : 4
Academic approval   : NOT CONFIRMED
Source modified     : NO
Training performed  : NO

OUTPUT ARTIFACTS:
- C:\Penelitian Sentimen Bencana Nasional\06_Result\outputs\final_technical_report\provisional_integrated_research_report.pdf
- C:\Penelitian Sentimen Bencana Nasional\06_Result\outputs\final_technical_report\final_pdf_figure_ma